In [ ]:
# Run this FIRST in the notebook's Python kernel (Python 3.11 or newer).
# These are all pipeline dependencies. No repository checkout or local .py files are needed.
# On Linux/Windows, pip's standard PyTorch wheels support CUDA where available.
# For a specific CUDA wheel, set TORCH_INDEX_URL using https://pytorch.org/get-started/locally/.
import sys
import subprocess

if sys.version_info < (3, 11):
    raise RuntimeError("Use a Python 3.11+ kernel for SciPy's Welch ANOVA support.")
TORCH_INDEX_URL = None  # e.g. a CUDA wheel index from the official PyTorch selector
if TORCH_INDEX_URL:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "torch>=2.6,<3",
                           "--index-url", TORCH_INDEX_URL])
subprocess.check_call([sys.executable, "-m", "pip", "install",
    "numpy>=1.26,<3", "pandas>=2.2,<4", "scipy>=1.16,<2", "scikit-learn>=1.6,<2",
    "torch>=2.6,<3", "xgboost>=3,<4", "openml>=0.15,<0.16", "threadpoolctl>=3.1,<4",
    "imbalanced-learn>=0.14,<0.15", "ipython>=8,<10"])
print("Dependencies installed. If an already imported package was upgraded, restart the kernel before continuing.")


# Standalone six-method oversampling comparison

This notebook contains the full executable pipeline. **Only this notebook is needed**; it embeds its Python source and materializes a private temporary package. Internet is needed for the first dependency installation and uncached OpenML datasets, not to download pipeline code.

## Dataset source and selection

The catalogue contains **32 binary CLIMB/imbalanced-ensemble entries**, retaining the original dataset numbers 1–30. Datasets **10** (Diabetes-130-Hospitals) and **26** (Click_prediction_small) are too large for this run and are skipped by default. Two small raw OpenML datasets are appended: **31 = haberman (306 rows, 3 predictors)** and **32 = blood-transfusion-service-center (748 rows, 4 predictors)**. `DATASET_RANGE="1-32"` with `EXCLUDED_DATASETS=[10, 26]` selects **30 active datasets**. To run only the additions, use `"31-32"`; other inclusive ranges such as `"4-9"` still work without renumbering. Set `EXCLUDED_DATASETS=[]` only if you deliberately want to allow the large datasets again. These are CLIMB sources, not imbalanced-learn's separate 27-dataset collection. Original OpenML binary targets are preserved, never silently relabelled one-versus-rest, and globally preprocessed package arrays are not used.

## Compared methods and classifiers

- `baseline`: train on real data only, no oversampling or class weighting.
- `original_ugo`: the paper-derived original UGOSMOTE implementation, with its documented ambiguity resolutions and synthetic-parent behavior. This is not corrected UGO or a verified author-code reproduction.
- `u2_flow_gate`: real-only KMeans/stability allocation, capped group/anchor quotas, native modified-margin difficulty, same-class real-parent interpolation, and frozen compatibility-only FlowGMM rejection. The gate replaces post-neighbor detection **and relocation**, rejects without refill, and uses equal priors. No historical U1 composition, downweighting, F2, utility selection or learned generator is included.
- `smote`: standard [imbalanced-learn SMOTE](https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.SMOTE.html), with a declared whole-category projection for mixed features (not silently substituted SMOTENC).
- `kmeans_smote`: [imbalanced-learn KMeansSMOTE](https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.KMeansSMOTE.html), with the same category projection. Its prespecified cluster-balance threshold is 0.0 so rare-positive clusters are eligible; clusters still need enough minority neighbors. Infeasible cases are recorded as failures, never replaced with another sampler.
- `rwo`: Random Walk Oversampling, not random duplication: uniform real-class anchors plus independent numeric Gaussian steps `std(ddof=0) / sqrt(n_real_class)`. Nominal variables use class-conditional frequency/roulette sampling of whole one-hot categories. Zero-variance numeric features remain constant; no clipping or rounding is added. Numeric convention follows the [Python RWO reference](https://github.com/analyticalmindsltd/smote_variants/blob/master/smote_variants/oversampling/_rwo_sampling.py); nominal handling follows the [RWO mixed-feature documentation](https://github.com/cran/imbalance/blob/master/R/rwo.R). The independently written notebook implementation is not a bitwise reproduction of that R implementation, whose numeric scaling convention differs.

**MLP, logistic regression, XGBoost, random forest and RBF SVM** are included. PyTorch MLP, FlowGMM and dense neighbor distances use CUDA when available, as does XGBoost. Standard sklearn logistic regression, random forest and SVM are CPU estimators; the notebook reports this explicitly rather than claiming GPU execution. `DEVICE="cpu"` works without an NVIDIA GPU. Driver/toolkit installation is a system prerequisite, not something pip can supply. See [PyTorch installation](https://pytorch.org/get-started/locally/).

SMOTE and KMeansSMOTE use their actual CPU imbalanced-learn implementations; vectorized RWO also runs on CPU. Their downstream MLP/XGBoost classifier still uses CUDA when selected. Resampling uses only the current real training role; validation and test observations are never oversampled. Fixed sampler parameters are editable below and are shared across classifiers. New controls use one-shot resampling, not UGO's iterative retraining.

## Evaluation and portability contract

Split real data first. All imputation/scaling/encoding fit inside the corresponding training role. Minority-class **F1** selects native model settings, U2 alpha, gate tau and the final classifier decision threshold using real inner validation only. AP is secondary. Equal F1 thresholds prefer closest to 0.5, then higher. The gate threshold `tau` and classifier decision threshold are distinct. SVM's probability estimation uses its fitting data, not the outer test. MLP uses fixed epochs, not validation-driven early stopping. The minority label is identified from real outer training, including label 0.

FlowGMM is fitted on a stratified 80% of real inner-fit rows; the remaining 20% is reserved and unused. Validation and outer test never fit the detector. Its preprocessing/weights remain frozen during generation and outer replay; only the downstream classifier/preprocessor refit on full outer training after selection. Synthetic observations never become U2 support or parents. Original UGO intentionally preserves its original descendant behavior. Category-safe partner projection for original UGO is a declared mixed-data adapter deviation.

The common synthetic target allowance is 1,000 rows, clipped at real majority balance. Original/gated UGO also bound proposals and may stop below that target; one-shot SMOTE/RWO aim at the declared final class counts. KMeansSMOTE can round cluster allocations upward: only excess synthetic rows are deterministically trimmed and recorded, never refilled or padded. U2 real-support caps and gate rejection can produce substantial shortfalls. Do not interpret a common allowance as count-matched generation quality. `MAX_ROUNDS` applies to UGO, not the three one-shot controls. Fixed 0.5 F1, precision, recall, AP, balanced accuracy, MCC, counts, native fits, runtime and backend are also reported. Seeds are repeated-split descriptive evidence, not independent-fold significance.

**Run a smoke first:** set `SMOKE=True` and a small range. It uses only original outer-training rows, shorter epochs and smaller budgets; its results are not scientific evidence. For the full study, use `SMOKE=False` and `MAX_ROWS=None`. Changing `MAX_ROWS` is an explicit development subsample and is recorded; no automatic subsampling occurs. All 30 datasets × 5 classifiers × 3 seeds × 6 methods means **2,700 reported scientific cells**, plus native tuning and internal canonical-U2 fits. RBF SVM and exact neighbors can be expensive on large datasets. Run separate numbered ranges when appropriate.

Outputs go to a unique directory under `OUTPUT_DIR`: `comparison.csv` shows all six methods side by side with F1 and seed coverage; `summary.csv` includes every metric; `paired_deltas.csv` includes all 15 unordered method pairs on matching splits. Compact predictions, selections, dataset/split identities and explicit failures are also retained. No trained model checkpoints, augmented arrays or raw proposal-coordinate logs are saved. Missing cells remain visibly incomplete, not fake metrics, and their means must not be ranked as complete-study results. Keep seeds/splits and configuration identical when combining range runs, and never average duplicate cells twice.

Cell runtime excludes the shared FlowGMM fit, which is recorded once per dataset/seed in `manifest.json` under `shared_flow_fits`; per-cell shared-fit seconds are references to that same fit and must not be summed across classifiers. The three new control pools are generated once per dataset/seed/role/method and reused across native candidates/classifiers: only the first consumer's cell wall time includes pool creation. Audit `shared_generation_seconds` is a shared reference, not a new fit every time it appears. Use manifest total elapsed time alongside fit counts rather than treating all cell timings as independently repeated end-to-end costs.

**Verification:** appended datasets 31/32 completed **60 CUDA smoke cells outside the repository**, covering both additions across all five classifiers and six methods. Six predictive metrics per cell, all 60 validation-selected thresholds and all 150 matched method pairs were independently recomputed. Five range/exclusion regressions passed, and original IDs 1–30 were checked unchanged. Earlier core evidence includes 60 CPU and 60 CUDA smoke cells, numerical sampler regressions, U2 native/gated/all-rejected parity and the original 30 source-target checks. The unchanged dependency cell was reused rather than reinstalled. The full 2,700-cell default study, clean-machine dependency installation and Windows/macOS execution have not been exercised.


In [ ]:
# EDIT THIS CELL. Numbering is 1-based and both endpoints are included.
DATASET_RANGE = "1-32"     # "31-32" runs only the two small additions; endpoints are inclusive
EXCLUDED_DATASETS = [10, 26]  # Keeps 30 active datasets without changing their original IDs
CLASSIFIERS = ["mlp", "logistic_regression", "xgboost", "random_forest", "svm"]
SEEDS = [42, 43, 44]
SMOKE = False              # True = short training-only integration check, NOT study results
DEVICE = "auto"            # "auto", "cpu", or "cuda" (explicit cuda requires working CUDA)
DATA_HOME = "ugo_data"      # Persistent raw OpenML cache; safe to reuse across ranges
OUTPUT_DIR = "ugo_results"
MAX_ROWS = None             # None uses every row; a number is an explicit development subset
N_CANDIDATES = 10
TEST_FRACTION = 0.2
VALIDATION_FRACTION = 0.2
AUGMENTATION_CAP = 1000
MAX_ROUNDS = 20
MLP_EPOCHS = 160
FLOW_EPOCHS = 60
GROUP_ALPHAS = [0.0, 0.5, 1.0]
GATE_TAUS = [0.5, 0.6, 0.7, 0.8]
N_JOBS = 4
# Prespecified sampler settings: no automatic neighbor weakening or sampler fallback.
SAMPLER_PARAMETERS = {
    "smote": {"k_neighbors": 5},
    "kmeans_smote": {"k_neighbors": 2, "kmeans_estimator": 8,
                     "cluster_balance_threshold": 0.0, "density_exponent": "auto"},
    "rwo": {},
}

CONFIG = {key: globals()[key] for key in (
    "DATASET_RANGE", "EXCLUDED_DATASETS", "CLASSIFIERS", "SEEDS", "SMOKE", "DEVICE", "DATA_HOME", "OUTPUT_DIR",
    "MAX_ROWS", "N_CANDIDATES", "TEST_FRACTION", "VALIDATION_FRACTION", "AUGMENTATION_CAP",
    "MAX_ROUNDS", "MLP_EPOCHS", "FLOW_EPOCHS", "GROUP_ALPHAS", "GATE_TAUS", "N_JOBS",
    "SAMPLER_PARAMETERS")}
# Optional programmatic overrides are useful for automated notebook execution.
CONFIG.update(globals().get("NOTEBOOK_OVERRIDES", {}))


## Bundled implementation

These readable source cells are part of the notebook; run them in order. They require no checkout, remote source download or sidecar modules. The core sampler sources are vendored from the tested repository implementation; preprocessing retains training-role isolation.


In [ ]:
BUNDLE_SOURCES = {}


In [ ]:
# Bundled module: __init__.py
BUNDLE_SOURCES['__init__.py'] = r'''"""Self-contained UGO notebook runtime."""
'''


In [ ]:
# Bundled module: preprocessing.py
BUNDLE_SOURCES['preprocessing.py'] = r'''import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler


def _normalize_missing(X):
    """Normalize nullable pandas values without learning from any partition."""
    result = X.copy()
    for column in result:
        values = result[column]
        if pd.api.types.is_numeric_dtype(values.dtype) and (not pd.api.types.is_bool_dtype(values.dtype)):
            result[column] = values.to_numpy(dtype=np.float64, na_value=np.nan)
            result[column] = result[column].replace([np.inf, -np.inf], np.nan)
        else:
            result[column] = values.map(lambda value: str(value) if pd.notna(value) else np.nan).astype(object)
    return result

def _dense_float32(X):
    return np.asarray(X, dtype=np.float32)

def make_preprocessor(X_train):
    """Build an unfitted transformer; fit it only on the actual training fold.

    Numeric features use median imputation and standardization. Categorical
    features use mode imputation and dense one-hot encoding; unseen categories
    become all-zero blocks. Empty training columns remain represented.
    """
    if not isinstance(X_train, pd.DataFrame):
        raise TypeError('make_preprocessor requires a raw pandas DataFrame.')
    numeric = [column for column in X_train if pd.api.types.is_numeric_dtype(X_train[column].dtype) and (not pd.api.types.is_bool_dtype(X_train[column].dtype))]
    categorical = [column for column in X_train if column not in numeric]
    columns = ColumnTransformer([('numeric', Pipeline([('impute', SimpleImputer(strategy='median', keep_empty_features=True)), ('scale', StandardScaler())]), numeric), ('categorical', Pipeline([('impute', SimpleImputer(strategy='most_frequent', keep_empty_features=True)), ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False, dtype=np.float32))]), categorical)], sparse_threshold=0)
    return Pipeline([('normalize', FunctionTransformer(_normalize_missing, feature_names_out='one-to-one')), ('columns', columns), ('float32', FunctionTransformer(_dense_float32, feature_names_out='one-to-one'))])

def categorical_columns(preprocessor):
    """Return one-hot output indices from an already fitted preprocessor."""
    block = preprocessor.named_steps['columns'].output_indices_['categorical']
    return tuple(range(block.start, block.stop))
'''


In [ ]:
# Bundled module: ugo.py
BUNDLE_SOURCES['ugo.py'] = r'''from collections.abc import Callable
from dataclasses import asdict, dataclass, field
from time import perf_counter
import numpy as np
import torch
from scipy.stats import f_oneway

@dataclass(frozen=True)
class UGOConfig:
    lambda_select: float = 0.5
    lambda_step: float = 0.1
    k_pre: int = 5
    k_after: int = 1
    phi: float = 2.5
    alpha: float = 0.05
    max_iterations: int = 20
    random_state: int = 42
    device: str = 'auto'
    categorical_columns: tuple[int, ...] = ()
    distance_batch_size: int = 512

    def __post_init__(self):
        if not 0 < self.lambda_select <= 1 or self.lambda_step <= 0 or self.phi <= 0:
            raise ValueError('Selection must be in (0, 1]; step and phi must be positive.')
        if min(self.k_pre, self.k_after, self.max_iterations, self.distance_batch_size) < 1:
            raise ValueError('Neighbor counts, iteration cap and distance batch size must be positive.')
        if not 0 <= self.alpha <= 1:
            raise ValueError('alpha must be in [0, 1].')
        if self.device not in ('auto', 'cpu', 'cuda'):
            raise ValueError('device must be auto, cpu or cuda.')

@dataclass
class RealSupport:
    X: torch.Tensor
    y: torch.Tensor
    classes: torch.Tensor
    eligible: torch.Tensor
    partners: dict[int, torch.Tensor]
    group_indices: dict[int, torch.Tensor]
    excluded_groups: list[int]

@dataclass
class GroupBudget:
    target_class: int
    seed_indices: torch.Tensor
    count: int

@dataclass
class Candidates:
    X: torch.Tensor
    y: torch.Tensor
    seed_indices: torch.Tensor
    partner_indices: torch.Tensor
    relocation_indices: torch.Tensor

@dataclass
class UGOResult:
    X: torch.Tensor
    y: torch.Tensor
    rounds: list[dict]
    provenance: list[dict]
    stop_reason: str
    config: UGOConfig
    support_summary: dict = field(default_factory=dict)

def nearest_indices(query, reference, k, batch_size=512, exclude_indices=None, exclude_zero=False):
    """Exact, bounded-memory Euclidean neighbors on the tensors' CPU/CUDA device."""
    if reference.device != query.device:
        raise ValueError('Query and reference must use the same device.')
    k = min(k, len(reference) - (exclude_indices is not None))
    if k <= 0:
        return torch.empty((len(query), 0), dtype=torch.long, device=query.device)
    result = torch.empty((len(query), k), dtype=torch.long, device=query.device)
    chunk = min(batch_size, max(1, 1000000 // len(reference)))
    for start in range(0, len(query), chunk):
        end = min(start + chunk, len(query))
        distances = torch.cdist(query[start:end], reference, compute_mode='donot_use_mm_for_euclid_dist')
        if exclude_zero:
            distances.masked_fill_(distances == 0, torch.inf)
        if exclude_indices is not None:
            distances[torch.arange(end - start, device=query.device), exclude_indices[start:end]] = torch.inf
        result[start:end] = distances.topk(k, dim=1, largest=False, sorted=True).indices
    return result

def _majority(neighbor_labels, class_count):
    counts = torch.zeros((len(neighbor_labels), class_count), dtype=torch.long, device=neighbor_labels.device)
    counts.scatter_add_(1, neighbor_labels, torch.ones_like(neighbor_labels))
    return counts.argmax(dim=1)

def noise_pre_detect(X: torch.Tensor, y: torch.Tensor, k: int=5, batch_size=512) -> torch.Tensor:
    if len(X) < 2:
        return torch.zeros(len(X), dtype=torch.bool, device=X.device)
    indices = nearest_indices(X, X, k, batch_size, torch.arange(len(X), device=X.device))
    return _majority(y[indices], int(y.max()) + 1) != y

def calculate_umma(probabilities: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """True-label modified margin difficulty, not an epistemic uncertainty estimate."""
    true_probabilities = probabilities.gather(1, y.unsqueeze(1)).squeeze(1)
    competing = probabilities.clone()
    competing.scatter_(1, y.unsqueeze(1), -torch.inf)
    return competing.max(dim=1).values - true_probabilities

def _welch_anova_p_value(uncertainty_groups: list[np.ndarray]) -> float:
    if any((len(group) < 2 for group in uncertainty_groups)):
        return 0.0
    constant = [np.allclose(group, group[0]) for group in uncertainty_groups]
    if all(constant):
        means = np.array([group[0] for group in uncertainty_groups])
        return 1.0 if np.allclose(means, means[0]) else 0.0
    if any(constant):
        return 0.0
    _, p_value = f_oneway(*uncertainty_groups, equal_var=False)
    return 0.0 if not np.isfinite(p_value) else float(p_value)

def predict_probabilities(model, X: torch.Tensor) -> torch.Tensor:
    if hasattr(model, 'predict_proba_tensor'):
        return model.predict_proba_tensor(X).to(device=X.device, dtype=X.dtype)
    if hasattr(model, 'predict_proba'):
        probabilities = model.predict_proba(X.detach().cpu().numpy())
        return torch.as_tensor(probabilities, dtype=X.dtype, device=X.device)
    model.eval()
    parameter = next(model.parameters())
    with torch.inference_mode():
        return torch.softmax(model(X.to(device=parameter.device, dtype=parameter.dtype)), dim=1).to(X.device)

def build_support(X, y, config):
    """Class-level support baseline; references and interpolation partners are real only."""
    classes = torch.unique(y, sorted=True)
    eligible = ~noise_pre_detect(X, y, config.k_pre, config.distance_batch_size)
    partners, groups, excluded_groups = ({}, {}, [])
    for label in classes.tolist():
        indices = torch.nonzero((y == label) & eligible, as_tuple=True)[0]
        if len(indices) < 2 or len(torch.unique(X[indices], dim=0)) < 2:
            eligible[indices] = False
            excluded_groups.append(label)
            continue
        groups[label] = indices
        local_neighbors = nearest_indices(X[indices], X[indices], min(config.k_pre, len(indices) - 1), config.distance_batch_size, torch.arange(len(indices), device=X.device), exclude_zero=True)
        partners[label] = indices[local_neighbors]
    return RealSupport(X, y, classes, eligible, partners, groups, excluded_groups)

def score_seeds(model, X, y):
    return calculate_umma(predict_probabilities(model, X), y)

def allocate_group_budget(support, scores, remaining, config):
    if not support.group_indices or remaining <= 0:
        return None
    target = max(support.group_indices, key=lambda label: float(scores[support.group_indices[label]].mean()))
    indices = support.group_indices[target]
    count = max(1, int(len(indices) * config.lambda_select))
    seeds = indices[torch.argsort(scores[indices], descending=True, stable=True)[:count]]
    return GroupBudget(target, seeds, min(remaining, max(1, round(len(support.y) * config.lambda_step))))

def propose_interpolation(support, budget, rng, config, *, seed_indices=None):
    device = support.X.device
    seeds = budget.seed_indices[torch.as_tensor(rng.integers(len(budget.seed_indices), size=budget.count), device=device)] if seed_indices is None else seed_indices
    group = support.group_indices[budget.target_class]
    local_seed = torch.searchsorted(group, seeds)
    options = support.partners[budget.target_class][local_seed]
    valid = (support.X[options] != support.X[seeds, None]).any(dim=2)
    valid_count = valid.sum(dim=1)
    keep = valid_count > 0
    seeds, options, valid, valid_count = (seeds[keep], options[keep], valid[keep], valid_count[keep])
    if len(seeds):
        choice = torch.as_tensor(rng.random(len(seeds)), device=device) * valid_count
        positions = (valid.cumsum(dim=1) > choice[:, None]).long().argmax(dim=1)
        partners = options[torch.arange(len(seeds), device=device), positions]
        interpolation = torch.as_tensor(rng.random((len(seeds), 1)), dtype=support.X.dtype, device=device)
        generated = support.X[seeds] + interpolation * (support.X[partners] - support.X[seeds])
        if config.categorical_columns:
            columns = list(config.categorical_columns)
            generated[:, columns] = support.X[partners][:, columns]
    else:
        partners = seeds.clone()
        generated = support.X[:0]
    return Candidates(generated, torch.full((len(seeds),), budget.target_class, dtype=support.y.dtype, device=device), seeds, partners, torch.full_like(seeds, -1))

def accept_relocation(support, candidates, config):
    if not len(candidates.X):
        return candidates
    nearest = nearest_indices(candidates.X, support.X, config.k_after, config.distance_batch_size)
    conflict = _majority(support.y[nearest], len(support.classes)) != candidates.y
    if conflict.any():
        target_indices = support.group_indices[int(candidates.y[0])]
        closest = nearest_indices(candidates.X[conflict], support.X[target_indices], 1, config.distance_batch_size)[:, 0]
        relocated = target_indices[closest]
        candidates.X[conflict] = support.X[relocated]
        candidates.relocation_indices[conflict] = relocated
    return candidates

def retrain_model(model_fn, X, y):
    return model_fn(X, y)

def stop_generation(p_value, generated, cap, config):
    if p_value > config.alpha:
        return 'class_difficulty_balance'
    if generated >= cap:
        return 'generation_cap'
    return None

class UGOPipeline:
    """Independent support → score → budget → propose → accept → retrain → stop stages.

    Callbacks use the public functions' contracts. Class-level allocation and original
    relocation are baselines, not the later group-protection/constraint experiments.
    """

    def __init__(self, config=None, *, support=build_support, seed_score=score_seeds, group_budget=allocate_group_budget, propose=propose_interpolation, accept=accept_relocation, retrain=retrain_model, stop=stop_generation):
        self.config = config or UGOConfig()
        self.support = support
        self.seed_score = seed_score
        self.group_budget = group_budget
        self.propose = propose
        self.accept = accept
        self.retrain = retrain
        self.stop = stop

    def run(self, X, y, model_fn: Callable, *, generation_cap=None):
        config = self.config
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') if config.device == 'auto' else torch.device(config.device)
        X = X.detach().to(device=device, dtype=torch.float32)
        y = y.detach().to(device=device, dtype=torch.long)
        classes = torch.unique(y, sorted=True)
        if X.ndim != 2 or y.ndim != 1 or len(X) != len(y) or (len(classes) < 2):
            raise ValueError('UGO requires a feature matrix and at least two classes.')
        if not torch.equal(classes, torch.arange(len(classes), device=device)):
            raise ValueError('Labels must be consecutive integers starting at zero.')
        if not torch.isfinite(X).all():
            raise ValueError('UGO requires finite, training-preprocessed features.')
        support = self.support(X, y, config)
        cap = max(0, int(config.phi * int(torch.bincount(y).max()) * len(classes)) - len(y))
        if generation_cap is not None:
            cap = min(cap, generation_cap)
        summary = {'real_count': len(y), 'eligible_seeds': int(support.eligible.sum()), 'excluded_seeds': int((~support.eligible).sum()), 'excluded_groups': support.excluded_groups, 'generation_cap': cap, 'device': str(device), 'configuration': asdict(config)}
        X_current, y_current = (X, y)
        rounds, provenance = ([], [])
        rng = np.random.default_rng(config.random_state)
        descendants = torch.zeros(len(y), dtype=torch.long, device=device)
        generated = 0
        stop_reason = 'iteration_cap'
        if not support.group_indices:
            return UGOResult(X, y, rounds, provenance, 'no_eligible_group', config, summary)
        for iteration in range(1, config.max_iterations + 1):
            if generated >= cap:
                stop_reason = 'generation_cap'
                break
            if device.type == 'cuda':
                torch.cuda.synchronize(device)
            start = perf_counter()
            model = self.retrain(model_fn, X_current, y_current)
            scores = self.seed_score(model, X, y)
            groups = [scores[(y == label) & support.eligible].detach().cpu().numpy() for label in classes]
            p_value = _welch_anova_p_value(groups)
            if device.type == 'cuda':
                torch.cuda.synchronize(device)
            fit_seconds = perf_counter() - start
            row = {'round': iteration, 'p_value': p_value, 'excluded_seeds': summary['excluded_seeds'], 'excluded_groups': summary['excluded_groups'], 'proposed': 0, 'accepted': 0, 'relocated': 0, 'unique_seeds': 0, 'duplicates': 0, 'retrains': iteration, 'fit_score_seconds': fit_seconds}
            row['class_mean_difficulty'] = {str(int(label)): float(group.mean()) if len(group) else None for label, group in zip(classes.tolist(), groups)}
            row['real_group_support'] = {str(int(label)): len(group) for label, group in zip(classes.tolist(), groups)}
            stop_reason_now = self.stop(p_value, generated, cap, config)
            if stop_reason_now:
                row.update(stop_reason=stop_reason_now, total_generated=generated, unspent_budget=cap - generated)
                rounds.append(row)
                stop_reason = stop_reason_now
                break
            budget = self.group_budget(support, scores, cap - generated, config)
            if budget is None:
                stop_reason = 'no_eligible_group'
                row['stop_reason'] = stop_reason
                rounds.append(row)
                break
            start = perf_counter()
            candidates = self.propose(support, budget, rng, config)
            proposed = len(candidates.X)
            candidates = self.accept(support, candidates, config)
            accepted = len(candidates.X)
            if not accepted:
                stop_reason = 'no_valid_partner'
                row.update(proposed=proposed, stop_reason=stop_reason)
                rounds.append(row)
                break
            old_unique = len(torch.unique(X_current, dim=0))
            offset = len(y_current)
            X_current = torch.cat((X_current, candidates.X))
            y_current = torch.cat((y_current, candidates.y))
            duplicates = accepted - (len(torch.unique(X_current, dim=0)) - old_unique)
            descendants.scatter_add_(0, candidates.seed_indices, torch.ones_like(candidates.seed_indices))
            for index, (seed, partner, relocation) in enumerate(zip(candidates.seed_indices.tolist(), candidates.partner_indices.tolist(), candidates.relocation_indices.tolist())):
                provenance.append({'row_index': offset + index, 'round': iteration, 'seed_index': seed, 'partner_index': partner, 'relocation_index': relocation, 'class': budget.target_class})
            generated += accepted
            if device.type == 'cuda':
                torch.cuda.synchronize(device)
            row.update(target_class=budget.target_class, proposed=proposed, accepted=accepted, relocated=int((candidates.relocation_indices >= 0).sum()), unique_seeds=len(torch.unique(candidates.seed_indices)), duplicates=duplicates, total_generated=generated, unspent_budget=cap - generated, descendants_per_seed={str(i): int(descendants[i]) for i in torch.nonzero(descendants, as_tuple=True)[0].tolist()}, generation_seconds=perf_counter() - start)
            rounds.append(row)
        if generated >= cap:
            stop_reason = 'generation_cap'
        return UGOResult(X_current, y_current, rounds, provenance, stop_reason, config, summary)
'''


In [ ]:
# Bundled module: ugo_original.py
BUNDLE_SOURCES['ugo_original.py'] = r'''"""Paper-derived numeric UGOSMOTE reference, NOT an executable author reproduction.

Audit: local 1-s2.0-S095070512600612X-main.pdf, Sağlam & Cengiz,
Knowledge-Based Systems 342 (2026) 115886, §§3.1–3.4, Tables 2–4.
Author code was not located in the paper or targeted public search.

Resolved ambiguities (not claims of exact author behavior):
* §3.1 explicitly retrains on augmented observations, whereas Table 4 leaves
  X,Y unchanged within the loop. Use the prose: descendants participate in
  in-sample scoring, neighbor support, interpolation and relocation.
* §3.3 cap is phi*max(initial class count)*K-n; Table 4 instead says
  phi*max(n_l). Use §3.3 with initial counts and clip the last batch.
* NoisePreDetect runs once (Table 4); generated/relocated rows are admitted
  as non-noisy. No real-only support restriction is imposed.
* R is a pluggable sampler; this implementation is UGOSMOTE: top-ranked
  seeds interpolate to k nearest current same-class observations. The paper
  does not specify neighbor-pool restriction or integer rounding: floor
  n_select and n_step, minimum one, exclude self, stable lower-class ties.
* Relocation copies the nearest non-noisy same-class observation, as §3.4
  specifies (not halfway interpolation). Undefined Welch tests halt with an
  explicit reason rather than treating an undefined statistic as evidence.

Operational max_iterations and optional target_counts are protocol deviations.
Optional categorical_projection copies the partner's categorical coordinates:
a controlled adapter deviation, disabled in faithful numeric mode. Numeric
mode can produce invalid categories; it must never be presented as type-safe.
"""
from dataclasses import asdict, dataclass
import numpy as np
import torch
from scipy.stats import f_oneway
from portable_ugo.ugo import UGOResult, calculate_umma, nearest_indices, predict_probabilities

@dataclass(frozen=True)
class OriginalUGOConfig:
    lambda_select: float = 0.5
    lambda_step: float = 0.1
    k_pre: int = 5
    k_after: int = 1
    k_neighbors: int = 5
    phi: float = 2.5
    alpha: float = 0.05
    max_iterations: int = 20
    random_state: int = 42
    device: str = 'auto'
    categorical_columns: tuple = ()
    categorical_projection: bool = False
    distance_batch_size: int = 512

    def __post_init__(self):
        if not 0 < self.lambda_select <= 1 or self.lambda_step <= 0 or self.phi <= 0:
            raise ValueError('Invalid UGO selection, step or phi')
        if min(self.k_pre, self.k_after, self.k_neighbors, self.max_iterations, self.distance_batch_size) < 1:
            raise ValueError('Neighbor counts and iteration cap must be positive')
        if not 0 <= self.alpha <= 1:
            raise ValueError('alpha must be in [0,1]')
        if self.device not in ('auto', 'cpu', 'cuda'):
            raise ValueError('Invalid UGO device')

def original_ugo(X, y, model_fn, config=None, target_counts=None):
    config = config or OriginalUGOConfig()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') if config.device == 'auto' else torch.device(config.device)
    X = torch.as_tensor(X, dtype=torch.float32, device=device).clone()
    y = torch.as_tensor(y, dtype=torch.long, device=device).clone()
    classes, counts = torch.unique(y, sorted=True, return_counts=True)
    if X.ndim != 2 or y.ndim != 1 or len(y) != len(X) or (len(classes) < 2) or (not bool(torch.isfinite(X).all())):
        raise ValueError('Original UGO requires finite features and at least two classes')
    if not torch.equal(classes, torch.arange(len(classes), device=device)):
        raise ValueError('Original UGO requires consecutive integer labels starting at zero')
    if len(y) <= config.k_pre or len(y) < config.k_after:
        raise ValueError('Original UGO has insufficient observations for requested noise neighbors')
    initial_n = len(y)
    cap = max(0, int(np.floor(config.phi * int(counts.max()) * len(classes) - initial_n)))
    targets = None if target_counts is None else {int(k): int(v) for k, v in target_counts.items()}
    if targets is not None:
        if set(targets) - set(classes.tolist()) or any((targets.get(int(c), int(n)) < int(n) for c, n in zip(classes, counts))):
            raise ValueError('target_counts must refer to observed classes and cannot remove rows')
        cap = min(cap, sum((targets.get(int(c), int(n)) - int(n) for c, n in zip(classes, counts))))
    own = torch.arange(initial_n, device=device)
    neighbors = nearest_indices(X, X, config.k_pre, config.distance_batch_size, own)
    eligible = torch.stack([y[row] == torch.bincount(y[ids], minlength=len(classes)).argmax() for row, ids in enumerate(neighbors)])
    rounds, provenance = ([], [])
    rng = np.random.default_rng(config.random_state)
    step = max(1, int(np.floor(initial_n * config.lambda_step)))
    generated = 0
    stop = 'iteration_cap'
    for iteration in range(1, config.max_iterations + 1):
        if generated >= cap:
            stop = 'generation_cap'
            break
        model = model_fn(X, y)
        scores = calculate_umma(predict_probabilities(model, X), y)
        groups = [torch.where(eligible & (y == c))[0] for c in classes]
        values = [scores[g].detach().cpu().numpy() for g in groups]
        row = {'round': iteration, 'training_count': len(y), 'eligible_count': int(eligible.sum()), 'synthetic_support_count': int(eligible[initial_n:].sum()), 'class_mean_difficulty': {int(c): float(v.mean()) if len(v) else None for c, v in zip(classes, values)}, 'generated': 0}
        if any((len(v) < 2 for v in values)) or any((float(np.var(v)) == 0 for v in values)):
            stop = 'undefined_welch_test'
            row['stop_reason'] = stop
            rounds.append(row)
            break
        p_value = float(f_oneway(*values, equal_var=False).pvalue)
        row['p_value'] = p_value if np.isfinite(p_value) else None
        if not np.isfinite(p_value) or p_value > config.alpha:
            stop = 'undefined_welch_test' if not np.isfinite(p_value) else 'class_difficulty_balance'
            row['stop_reason'] = stop
            rounds.append(row)
            break
        available = [int(c) for c in classes if targets is None or int((y == c).sum()) < targets.get(int(c), int((y == c).sum()))]
        if not available:
            stop = 'target_counts_reached'
            break
        c = max(available, key=lambda label: float(scores[groups[label]].mean()))
        group = groups[c]
        selected_n = max(1, int(np.floor(len(group) * config.lambda_select)))
        selected = group[torch.argsort(scores[group], descending=True, stable=True)[:selected_n]]
        all_class = torch.where(y == c)[0]
        if len(all_class) <= config.k_neighbors:
            raise ValueError(f'Original UGOSMOTE class {c} needs more than {config.k_neighbors} observations')
        options = nearest_indices(X[selected], X[all_class], config.k_neighbors, config.distance_batch_size, torch.searchsorted(all_class, selected))
        count = min(step, cap - generated)
        if targets is not None:
            count = min(count, targets.get(c, int((y == c).sum())) - int((y == c).sum()))
        seed_local = rng.integers(len(selected), size=count)
        seeds = selected[torch.as_tensor(seed_local, device=device)]
        partner_local = rng.integers(config.k_neighbors, size=count)
        partners = all_class[options[torch.as_tensor(seed_local, device=device), torch.as_tensor(partner_local, device=device)]]
        fraction = torch.as_tensor(rng.random((count, 1)), dtype=X.dtype, device=device)
        candidates = X[seeds] + fraction * (X[partners] - X[seeds])
        if config.categorical_projection and config.categorical_columns:
            columns = list(config.categorical_columns)
            candidates[:, columns] = X[partners][:, columns]
        nearby = nearest_indices(candidates, X, config.k_after, config.distance_batch_size)
        majority = torch.stack([torch.bincount(y[ids], minlength=len(classes)).argmax() for ids in nearby])
        conflict = majority != c
        relocation = torch.full((count,), -1, device=device, dtype=torch.long)
        if bool(conflict.any()):
            nearest = nearest_indices(candidates[conflict], X[group], 1, config.distance_batch_size)[:, 0]
            relocation[conflict] = group[nearest]
            candidates[conflict] = X[relocation[conflict]]
        offset = len(y)
        for index in range(count):
            provenance.append({'row_index': offset + index, 'round': iteration, 'class': c, 'seed_index': int(seeds[index]), 'partner_index': int(partners[index]), 'relocation_index': int(relocation[index]), 'seed_is_synthetic': bool(seeds[index] >= initial_n), 'partner_is_synthetic': bool(partners[index] >= initial_n), 'categorical_projection': config.categorical_projection})
        X = torch.cat((X, candidates))
        y = torch.cat((y, torch.full((count,), c, device=device, dtype=y.dtype)))
        eligible = torch.cat((eligible, torch.ones(count, device=device, dtype=torch.bool)))
        generated += count
        row.update(target_class=c, selected_count=selected_n, generated=count, relocated=int(conflict.sum()), total_generated=generated)
        rounds.append(row)
    if generated >= cap:
        stop = 'generation_cap'
    deviations = ['operational_iteration_cap', 'paper_prose_augmented_retraining_resolution', 'section_3_3_initial_count_cap_resolution', 'floor_rounding_and_self_exclusion_resolution']
    if targets is not None:
        deviations.append('matched_per_class_target_caps')
    if config.categorical_projection:
        deviations.append('categorical_partner_projection')
    summary = {'implementation': 'paper_derived_numeric_UGOSMOTE', 'author_reproduction_status': 'unresolved', 'audit_source': '1-s2.0-S095070512600612X-main.pdf §§3.1–3.4 Tables 2–4', 'real_count': initial_n, 'generation_cap': cap, 'target_counts': targets, 'actual_counts': {int(c): int((y == c).sum()) for c in classes}, 'protocol_deviations': deviations, 'configuration': asdict(config)}
    return UGOResult(X, y, rounds, provenance, stop, config, summary)
'''


In [ ]:
# Bundled module: groups.py
BUNDLE_SOURCES['groups.py'] = r'''"""Real-only subgroup support and subsampling co-membership reliability.

Groups describe eligible support, not proposal or relocation geometry. Stability
measures splitting/merging under real-row subsampling; it is not a noise score.
"""
from dataclasses import asdict, dataclass
from numbers import Integral, Real
import numpy as np
from scipy.sparse import csr_matrix, vstack
from scipy.sparse.csgraph import connected_components
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors

@dataclass(frozen=True)
class GroupConfig:
    grouping: str = 'kmeans'
    n_clusters: int = 8
    n_neighbors: int = 5
    repeats: int = 5
    sample_fraction: float = 0.8
    radius_quantile: float = 0.9

    def __post_init__(self):
        if self.grouping not in ('kmeans', 'graph'):
            raise ValueError('grouping must be kmeans or graph')
        for name in ('n_clusters', 'n_neighbors', 'repeats'):
            value = getattr(self, name)
            if isinstance(value, (bool, np.bool_)) or not isinstance(value, Integral) or value < 1:
                raise ValueError(f'{name} must be a positive integer')
        for name, lower_open in (('sample_fraction', True), ('radius_quantile', False)):
            value = getattr(self, name)
            if isinstance(value, (bool, np.bool_)) or not isinstance(value, Real) or (not np.isfinite(value)) or (not 0 <= value <= 1) or (lower_open and value == 0):
                interval = '(0, 1]' if lower_open else '[0, 1]'
                raise ValueError(f'{name} must be finite and in {interval}')

def _graph_labels(coordinates, config):
    """Sparse mutual kNN, with bounded-memory neighbor queries."""
    count = len(coordinates)
    if count == 1:
        return np.zeros(1, dtype=np.int64)
    k = min(config.n_neighbors, count - 1)
    neighbors = NearestNeighbors(n_neighbors=k + 1, algorithm='kd_tree').fit(coordinates)
    batch_size = max(1, min(512, 1000000 // (k + 1)))
    kth_distances = np.empty(count, dtype=float)
    for start in range(0, count, batch_size):
        stop = min(start + batch_size, count)
        distances, _ = neighbors.kneighbors(coordinates[start:stop])
        kth_distances[start:stop] = distances[:, -1]
    radius = np.quantile(kth_distances, config.radius_quantile)
    blocks = []
    for start in range(0, count, batch_size):
        stop = min(start + batch_size, count)
        distances, indices = neighbors.kneighbors(coordinates[start:stop])
        retained = distances[:, 1:] <= radius
        row, column = np.nonzero(retained)
        blocks.append(csr_matrix((np.ones(len(row), dtype=bool), (row, indices[:, 1:][row, column])), shape=(stop - start, count)))
    directed = vstack(blocks, format='csr')
    mutual = directed.multiply(directed.T)
    return connected_components(mutual, directed=False, return_labels=True)[1]

def _fit_groups(rows, seed, config):
    """Deduplicate for geometry, then restore every real observation."""
    coordinates, inverse = np.unique(rows, axis=0, return_inverse=True)
    if config.grouping == 'kmeans':
        unique_labels = KMeans(n_clusters=min(config.n_clusters, len(coordinates)), n_init=10, random_state=seed).fit_predict(coordinates)
    else:
        unique_labels = _graph_labels(coordinates, config)
    row_labels = unique_labels[inverse]
    labels, first_indices = np.unique(row_labels, return_index=True)
    canonical = labels[np.argsort(first_indices)]
    remap = np.empty(int(unique_labels.max()) + 1, dtype=np.int64)
    remap[canonical] = np.arange(len(canonical))
    return (remap[row_labels], np.bincount(remap[unique_labels], minlength=len(canonical)))

def _co_membership_jaccard(base_labels, replica_labels, n_groups):
    """Count incident pairs algebraically, including replica-cluster outsiders.

    Both label arrays cover the same observed real rows. No pair sets or dense
    co-membership matrix are constructed.
    """
    observed = np.bincount(base_labels, minlength=n_groups)
    evaluable = observed >= 2
    ratios = np.zeros(n_groups, dtype=float)
    if not len(base_labels):
        return (ratios, evaluable)
    replica_counts = np.bincount(replica_labels)
    replica_count = len(replica_counts)
    cells, members = np.unique(base_labels * replica_count + replica_labels, return_counts=True)
    group_ids, replica_ids = np.divmod(cells, replica_count)
    intersection = np.zeros(n_groups, dtype=np.int64)
    incident = np.zeros(n_groups, dtype=np.int64)
    np.add.at(intersection, group_ids, members * (members - 1) // 2)
    np.add.at(incident, group_ids, members * (2 * replica_counts[replica_ids] - members - 1) // 2)
    base_pairs = observed * (observed - 1) // 2
    union = base_pairs + incident - intersection
    ratios[evaluable] = intersection[evaluable] / union[evaluable]
    return (ratios, evaluable)

def prepare_groups(X, y, target_labels, seed, config):
    """Fit real support groups and independently refit real-row subsamples.

    Subsampling is without replacement within each target class, using
    ceil(sample_fraction * real_support) rows. Duplicate coordinates contribute
    their original row multiplicity to support and stability, never extra
    coordinates. Groups with fewer than two distinct coordinates are explicit
    but cannot provide interpolative support.
    """
    if not isinstance(config, GroupConfig):
        raise TypeError('config must be a GroupConfig')
    X, y = (np.asarray(X), np.asarray(y))
    if X.ndim != 2 or X.shape[1] == 0 or y.ndim != 1 or (len(X) != len(y)) or (not np.issubdtype(X.dtype, np.number)) or (not np.isfinite(X).all()):
        raise ValueError('Grouping requires finite real features and matching one-dimensional labels')
    target_labels = np.asarray(list(target_labels))
    rng = np.random.default_rng(seed)
    groups = []
    for label in np.unique(y):
        if not np.any(target_labels == label):
            continue
        source_indices = np.flatnonzero(y == label)
        rows = X[source_indices]
        base_labels, distinct_counts = _fit_groups(rows, seed, config)
        n_groups = len(distinct_counts)
        stability_sum = np.zeros(n_groups, dtype=float)
        evaluable_counts = np.zeros(n_groups, dtype=np.int64)
        sample_size = int(np.ceil(config.sample_fraction * len(rows)))
        for _ in range(config.repeats):
            sampled = np.sort(rng.choice(len(rows), size=sample_size, replace=False))
            repeat_seed = int(rng.integers(0, np.iinfo(np.uint32).max))
            replica_labels, _ = _fit_groups(rows[sampled], repeat_seed, config)
            ratios, evaluable = _co_membership_jaccard(base_labels[sampled], replica_labels, n_groups)
            stability_sum += ratios
            evaluable_counts += evaluable
        stability = np.divide(stability_sum, evaluable_counts, out=np.zeros(n_groups), where=evaluable_counts > 0)
        for group_id in range(n_groups):
            indices = source_indices[base_labels == group_id]
            groups.append({'id': -1, 'class': label.item(), 'indices': indices, 'real_support': int(len(indices)), 'distinct_coordinates': int(distinct_counts[group_id]), 'stability': float(stability[group_id]), 'evaluable_repeats': int(evaluable_counts[group_id])})
    groups.sort(key=lambda group: int(group['indices'][0]))
    row_group_ids = np.full(len(y), -1, dtype=np.int64)
    for group_id, group in enumerate(groups):
        group['id'] = group_id
        row_group_ids[group['indices']] = group_id
    return {'row_group_ids': row_group_ids, 'groups': groups, 'configuration': asdict(config), 'synthetic_training_count': 0}
'''


In [ ]:
# Bundled module: allocation.py
BUNDLE_SOURCES['allocation.py'] = r'''"""Real-only U2 quota planning; allocation does not change UGO geometry."""
from dataclasses import dataclass
from math import ceil, floor
from numbers import Integral
import numpy as np
from scipy.stats import rankdata

@dataclass(frozen=True)
class AllocationConfig:
    allocation: str = 'stability'
    group_alpha: float = 0.5
    epsilon: float = 0.001
    seed_cap: int = 4
    group_cap_multiplier: float = 2.0

    def __post_init__(self):
        if self.allocation not in ('stability', 'size', 'uniform', 'difficulty'):
            raise ValueError('allocation must be stability, size, uniform or difficulty')
        if not np.isfinite(self.group_alpha) or not 0 <= self.group_alpha <= 1:
            raise ValueError('group_alpha must be finite and in [0, 1]')
        if not np.isfinite(self.epsilon) or self.epsilon <= 0:
            raise ValueError('epsilon must be finite and positive')
        if isinstance(self.seed_cap, bool) or not isinstance(self.seed_cap, Integral) or self.seed_cap < 1:
            raise ValueError('seed_cap must be a positive integer')
        if not np.isfinite(self.group_cap_multiplier) or self.group_cap_multiplier <= 0:
            raise ValueError('group_cap_multiplier must be finite and positive')

def integer_quotas(weights, capacities, budget):
    """Capped proportional largest remainders, redistributing saturated shares.

    Zero weights never receive quota, including when positive-weight capacities
    are exhausted. Equal remainders are resolved in original input order.
    """
    weights = np.asarray(weights, dtype=float)
    capacities = np.asarray(capacities)
    if weights.ndim != 1 or capacities.shape != weights.shape:
        raise ValueError('weights and capacities must be matching vectors')
    if not np.all(np.isfinite(weights)) or np.any(weights < 0):
        raise ValueError('weights must be finite and nonnegative')
    if capacities.size and (not np.issubdtype(capacities.dtype, np.integer) or np.any(capacities < 0)):
        raise ValueError('capacities must be nonnegative integers')
    if isinstance(budget, bool) or not isinstance(budget, Integral) or budget < 0:
        raise ValueError('budget must be a nonnegative integer')
    capacities = capacities.astype(np.int64, copy=False)
    quotas = np.zeros(len(weights), dtype=np.int64)
    eligible = np.flatnonzero((weights > 0) & (capacities > 0))
    remaining = min(int(budget), sum((int(capacities[i]) for i in eligible)))
    while remaining and len(eligible):
        normalized = weights[eligible] / weights[eligible].max()
        shares = remaining * (normalized / normalized.sum())
        room = capacities[eligible] - quotas[eligible]
        saturated = shares >= room
        if np.any(saturated):
            filled = eligible[saturated]
            quotas[filled] = capacities[filled]
            remaining -= sum((int(n) for n in room[saturated]))
            eligible = eligible[~saturated]
            continue
        whole = np.floor(shares).astype(np.int64)
        quotas[eligible] += whole
        remaining -= sum((int(n) for n in whole))
        order = np.argsort(-(shares - whole), kind='stable')
        quotas[eligible[order[:remaining]]] += 1
        remaining = 0
    return quotas

class GroupAllocator:
    """Plan repeated source seeds and account only accepted descendants.

    ``prepared_groups`` is the real-only output of ``prepare_groups``. Final
    class targets, seed caps and group caps remain fixed across every commit.
    Planning itself never reserves or consumes any capacity.
    """

    def __init__(self, prepared_groups, y, target_counts=None, config=None):
        self.config = config or AllocationConfig()
        self.y = np.asarray(y)
        if self.y.ndim != 1 or not len(self.y):
            raise ValueError('y must be a nonempty vector of real source labels')
        classes, counts = np.unique(self.y, return_counts=True)
        self.real_counts = {c.item(): int(n) for c, n in zip(classes, counts)}
        supplied = {} if target_counts is None else dict(target_counts)
        if any((isinstance(v, bool) or not isinstance(v, Integral) for v in supplied.values())):
            raise ValueError('target_counts must contain integer final counts')
        if set(supplied) - set(self.real_counts):
            raise ValueError('target_counts cannot introduce classes')
        self.target_counts = {c: int(counts.max()) if target_counts is None else int(supplied.get(c, n)) for c, n in self.real_counts.items()}
        if any((self.target_counts[c] < n for c, n in self.real_counts.items())):
            raise ValueError('target_counts cannot remove real observations')
        self.groups = prepared_groups['groups']
        self.used_seed = np.zeros(len(self.y), dtype=np.int64)
        self.used_group = np.zeros(len(self.groups), dtype=np.int64)
        self.used_class = {c: 0 for c in self.real_counts}
        self.records = []
        self._seed_group = np.full(len(self.y), -1, dtype=np.int64)
        self._group_caps = np.zeros(len(self.groups), dtype=np.int64)
        self._eligible = np.zeros(len(self.groups), dtype=bool)
        seen_ids = set()
        for pos, group in enumerate(self.groups):
            indices = np.asarray(group['indices'])
            if indices.ndim != 1 or (indices.size and (not np.issubdtype(indices.dtype, np.integer))):
                raise ValueError('group indices must be an integer vector')
            if np.any(indices < 0) or np.any(indices >= len(self.y)) or len(np.unique(indices)) != len(indices):
                raise ValueError('groups must contain distinct real source indices')
            if group['id'] in seen_ids or np.any(self._seed_group[indices] != -1):
                raise ValueError('group IDs and source memberships must be unique')
            seen_ids.add(group['id'])
            if group['class'] not in self.real_counts or np.any(self.y[indices] != group['class']):
                raise ValueError('group class must match its real source labels')
            n = group['real_support']
            if not isinstance(n, Integral) or n != len(indices):
                raise ValueError('real_support must equal the real source membership count')
            distinct = group['distinct_coordinates']
            if not isinstance(distinct, Integral) or not 0 <= distinct <= n:
                raise ValueError('distinct_coordinates must be an integer bounded by real_support')
            stability = group['stability']
            if not np.isfinite(stability) or not 0 <= stability <= 1:
                raise ValueError('stability must be finite and in [0, 1]')
            self._seed_group[indices] = pos
            self._group_caps[pos] = floor(self.config.group_cap_multiplier * n)
            self._eligible[pos] = n >= 2 and distinct >= 2 and (stability > 0)

    def allocate(self, scores, target_class, budget, selection_fraction):
        scores = np.asarray(scores, dtype=float)
        if scores.shape != self.y.shape or not np.all(np.isfinite(scores)):
            raise ValueError('scores must be one finite score per real source row')
        if target_class not in self.real_counts:
            raise ValueError('target_class must be an observed real class')
        if isinstance(budget, bool) or not isinstance(budget, Integral) or budget < 0:
            raise ValueError('budget must be a nonnegative integer')
        if not np.isfinite(selection_fraction) or not 0 < selection_fraction <= 1:
            raise ValueError('selection_fraction must be finite and in (0, 1]')
        class_indices = np.flatnonzero(self.y == target_class)
        percentiles = np.zeros(len(self.y), dtype=float)
        percentiles[class_indices] = (rankdata(scores[class_indices], method='average') - 0.5) / len(class_indices)
        remaining = self.target_counts[target_class] - self.real_counts[target_class] - self.used_class[target_class]
        weights = np.zeros(len(self.groups), dtype=float)
        capacities = np.zeros(len(self.groups), dtype=np.int64)
        selected = {}
        trace = []
        for pos, group in enumerate(self.groups):
            if group['class'] != target_class:
                continue
            indices = np.asarray(group['indices'], dtype=np.int64)
            n = group['real_support']
            h = float(percentiles[indices].mean()) if n else 0.0
            reasons = []
            if n < 2:
                reasons.append('insufficient_real_support')
            if group['distinct_coordinates'] < 2:
                reasons.append('insufficient_distinct_coordinates')
            if group['stability'] <= 0:
                reasons.append('zero_stability')
            if not reasons:
                if self.config.allocation == 'stability':
                    weights[pos] = group['stability'] * (h + self.config.epsilon) / (n + self.config.epsilon) ** self.config.group_alpha
                elif self.config.allocation == 'size':
                    weights[pos] = n
                elif self.config.allocation == 'uniform':
                    weights[pos] = 1.0
                else:
                    weights[pos] = h + self.config.epsilon
                available = indices[self.used_seed[indices] < self.config.seed_cap]
                order = np.argsort(-scores[available], kind='stable')
                chosen = available[order[:ceil(selection_fraction * n)]]
                selected[pos] = chosen
                seed_room = int((self.config.seed_cap - self.used_seed[chosen]).sum())
                group_room = int(self._group_caps[pos] - self.used_group[pos])
                capacities[pos] = min(seed_room, group_room, remaining)
                if seed_room == 0:
                    reasons.append('seed_cap_exhausted')
                if group_room == 0:
                    reasons.append('group_cap_exhausted')
                if remaining == 0:
                    reasons.append('class_target_reached')
            trace.append({'group_id': group['id'], 'class': group['class'], 'n_real': n, 'stability': float(group['stability']), 'h': h, 'weight': float(weights[pos]), 'capacity': int(capacities[pos]), 'quota': 0, 'used_group': int(self.used_group[pos]), 'used_seed': {int(i): int(self.used_seed[i]) for i in indices}, 'exclusions': reasons})
        quotas = integer_quotas(weights, capacities, min(int(budget), remaining))
        plans = []
        for pos, group in enumerate(self.groups):
            quota = int(quotas[pos])
            if quota:
                seeds = selected[pos]
                seed_counts = integer_quotas(np.ones(len(seeds)), self.config.seed_cap - self.used_seed[seeds], quota)
                plans.append(np.repeat(seeds, seed_counts))
        for entry, pos in zip(trace, (i for i, g in enumerate(self.groups) if g['class'] == target_class)):
            entry['quota'] = int(quotas[pos])
        seeds = np.concatenate(plans) if plans else np.empty(0, dtype=np.int64)
        record = {'target_class': target_class, 'allocation': self.config.allocation, 'requested': int(budget), 'allocated': len(seeds), 'unspent': int(budget) - len(seeds), 'class_remaining': remaining, 'used_class': self.used_class[target_class], 'accepted': 0, 'group_trace': trace}
        self.records.append(record)
        if not np.any((capacities > 0) & (weights > 0)):
            return None
        return {'target_class': target_class, 'seed_indices': seeds, 'count': len(seeds), 'group_trace': trace}

    def commit(self, seed_indices):
        """Atomically consume caps for accepted real-source seeds only."""
        seeds = np.asarray(seed_indices)
        if seeds.ndim != 1 or (seeds.size and (not np.issubdtype(seeds.dtype, np.integer))):
            raise ValueError('accepted seeds must be an integer vector')
        if not len(seeds):
            return
        if np.any(seeds < 0) or np.any(seeds >= len(self.y)):
            raise ValueError('accepted seeds must refer to real source rows')
        groups = self._seed_group[seeds]
        if np.any(groups < 0) or not np.all(self._eligible[groups]):
            raise ValueError('accepted seeds must belong to supported stable groups')
        classes = np.unique(self.y[seeds])
        if len(classes) != 1:
            raise ValueError('one commit must contain a single target class')
        target_class = classes[0].item()
        seed_delta = np.bincount(seeds, minlength=len(self.y))
        group_delta = np.bincount(groups, minlength=len(self.groups))
        if np.any(self.used_seed + seed_delta > self.config.seed_cap):
            raise ValueError('accepted seeds exceed the per-seed cap')
        if np.any(self.used_group + group_delta > self._group_caps):
            raise ValueError('accepted seeds exceed the group cap')
        if self.real_counts[target_class] + self.used_class[target_class] + len(seeds) > self.target_counts[target_class]:
            raise ValueError('accepted seeds exceed the final class target')
        self.used_seed += seed_delta
        self.used_group += group_delta
        self.used_class[target_class] += len(seeds)
        if self.records and self.records[-1]['target_class'] == target_class:
            self.records[-1]['accepted'] += len(seeds)
'''


In [ ]:
# Bundled module: flow_gmm.py
BUNDLE_SOURCES['flow_gmm.py'] = r'''"""Independently written, pinned ``flowgmm_tabular_v1`` F1 compatibility model.

Algorithm reference only: Izmailov et al., *Semi-Supervised Learning with
Normalizing Flows*, ICML 2020, https://proceedings.mlr.press/v119/izmailov20a.html;
https://github.com/izmailovpavel/flowgmm commit
071285be739defd548ed0c90eec9e56e1112b773. No upstream code is vendored (no
license was detected). This implementation is supervised, real-only and binary.

Full one-hot coordinates receive independent U[0,1) dequantization. Training
jitter is fresh every epoch; evaluation jitter is content-keyed, label-independent
and invariant to row order, duplicates and batching. Numeric features, including
counts, remain continuous, as in U2 interpolation. Reported density is a
continuous dequantized proxy in the detector's standardized/one-hot coordinates,
NOT an exact discrete probability, support guarantee or calibrated correctness
probability. F1 uses equal priors and no likelihood support filter.
"""
from dataclasses import asdict, dataclass
import hashlib
import math
from numbers import Integral
from pathlib import Path
from time import perf_counter
import numpy as np
import pandas as pd
import torch
from torch import nn
from portable_ugo.preprocessing import make_preprocessor
IMPLEMENTATION_VERSION = 'flowgmm_tabular_v1'
REFERENCE_COMMIT = '071285be739defd548ed0c90eec9e56e1112b773'

@dataclass(frozen=True)
class FlowGMMConfig:
    epochs: int = 60
    layers: int = 6
    hidden: int = 64
    batch_size: int = 256
    learning_rate: float = 0.001
    device: str = 'auto'

    def __post_init__(self):
        for name in ('epochs', 'layers', 'hidden', 'batch_size'):
            value = getattr(self, name)
            if isinstance(value, (bool, np.bool_)) or not isinstance(value, Integral) or value < 1:
                raise ValueError(f'{name} must be a positive integer')
        if isinstance(self.learning_rate, (bool, np.bool_)) or not np.isfinite(self.learning_rate) or self.learning_rate <= 0:
            raise ValueError('learning_rate must be positive and finite')
        if self.device not in ('auto', 'cpu', 'cuda'):
            raise ValueError('device must be auto, cpu or cuda')

def _layout(preprocessor):
    columns = preprocessor.named_steps['columns']
    numeric = list(columns.transformers_[0][2])
    categorical = list(columns.transformers_[1][2])
    encoder = columns.named_transformers_['categorical'].named_steps['encode'] if categorical else None
    start = columns.output_indices_['categorical'].start
    groups = []
    if encoder is not None:
        for categories in encoder.categories_:
            groups.append(tuple(range(start, start + len(categories))))
            start += len(categories)
    return (numeric, categorical, encoder, groups)

class AffineCoupling(nn.Module):
    """Alternating affine coupling, x -> z, with log scale 2*tanh(raw)."""

    def __init__(self, width, hidden, parity):
        super().__init__()
        self.register_buffer('mask', (torch.arange(width) % 2 == parity).double())
        self.network = nn.Sequential(nn.Linear(width, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 2 * width))
        nn.init.zeros_(self.network[-1].weight)
        nn.init.zeros_(self.network[-1].bias)

    def _parameters_at(self, fixed):
        raw_scale, shift = self.network(fixed).chunk(2, dim=-1)
        active = 1 - self.mask
        return (2 * torch.tanh(raw_scale) * active, shift * active)

    def forward(self, x):
        fixed = x * self.mask
        scale, shift = self._parameters_at(fixed)
        return (fixed + (1 - self.mask) * (x * torch.exp(scale) + shift), scale.sum(-1))

    def inverse(self, z):
        fixed = z * self.mask
        scale, shift = self._parameters_at(fixed)
        return (fixed + (1 - self.mask) * ((z - shift) * torch.exp(-scale)), -scale.sum(-1))

class SharedRealNVP(nn.Module):
    """One invertible map and two learned diagonal Gaussian latent components."""

    def __init__(self, width, config, means, stds):
        super().__init__()
        self.couplings = nn.ModuleList((AffineCoupling(width, config.hidden, i % 2) for i in range(config.layers)))
        self.means = nn.Parameter(torch.as_tensor(means, dtype=torch.float64).clone())
        self.log_stds = nn.Parameter(torch.as_tensor(stds, dtype=torch.float64).log())
        self.double()

    def forward(self, x):
        determinant = torch.zeros(x.shape[:-1], dtype=x.dtype, device=x.device)
        for coupling in self.couplings:
            x, increment = coupling(x)
            determinant = determinant + increment
        return (x, determinant)

    def inverse(self, z):
        determinant = torch.zeros(z.shape[:-1], dtype=z.dtype, device=z.device)
        for coupling in reversed(self.couplings):
            z, increment = coupling.inverse(z)
            determinant = determinant + increment
        return (z, determinant)

    def gaussian_log_density(self, z):
        z = z.double()
        standardized = (z[:, None, :] - self.means.double()) * torch.exp(-self.log_stds.double())
        return (-0.5 * standardized.square() - self.log_stds.double() - 0.5 * math.log(2 * math.pi)).sum(-1)

    def gaussian_log_ratio(self, z, intended):
        standardized = (z.double()[:, None, :] - self.means.double()) * torch.exp(-self.log_stds.double())
        rows = torch.arange(len(z), device=z.device)
        own, opposing = (standardized[rows, intended], standardized[rows, 1 - intended])
        return (0.5 * (opposing - own) * (opposing + own) + self.log_stds[1 - intended] - self.log_stds[intended]).sum(-1)

def _source_sha():
    return hashlib.sha256(Path(__file__).read_bytes()).hexdigest()

def _checksum(model):
    digest = hashlib.sha256()
    for name, tensor in model.state_dict().items():
        digest.update(name.encode())
        digest.update(tensor.detach().cpu().numpy().tobytes())
    return digest.hexdigest()

class FlowGMMCore:
    """Frozen detector; forward/inverse operate on detector coordinates (no jitter)."""

    def __init__(self, model, preprocessor, classes, fit_row_ids, config, seed, audit):
        self.model, self.preprocessor = (model, preprocessor)
        self.classes = np.asarray(classes).copy()
        self.fit_row_ids = np.asarray(fit_row_ids).copy()
        self.classes.setflags(write=False)
        self.fit_row_ids.setflags(write=False)
        self.config, self.seed, self.audit = (config, int(seed), audit)
        _, _, _, groups = _layout(preprocessor)
        self.categorical_groups = groups
        self.categorical_columns = tuple((column for group in groups for column in group))
        self.model.eval()
        self.model.requires_grad_(False)
        self.model.zero_grad(set_to_none=True)

    @property
    def device(self):
        return self.model.means.device

    def _tensor(self, X):
        return torch.as_tensor(X, dtype=torch.float64, device=self.device)

    def forward(self, X):
        return self.model(self._tensor(X))

    def inverse(self, Z):
        return self.model.inverse(self._tensor(Z))

    def represent(self, frame):
        """Fit-only standardized/full-one-hot coordinates, then deterministic jitter.

        Unknown categories raise explicitly, rather than turning all-zero blocks
        into a fabricated numeric category. Gate isolates row-specific rejections.
        """
        X = np.asarray(self.preprocessor.transform(frame), dtype=np.float64)
        for group in self.categorical_groups:
            if not np.all(X[:, group].sum(axis=1) == 1):
                raise ValueError('unknown_flow_category')
        if not np.isfinite(X).all():
            raise ValueError('nonfinite_score')
        return self._dequantize(X)

    def _dequantize(self, X):
        """Jitter an owned, already-validated detector-coordinate array in place."""
        if self.categorical_columns:
            for i, row in enumerate(X):
                key = hashlib.sha256(int(self.seed).to_bytes(8, 'little', signed=False) + row.astype('<f8', copy=False).tobytes()).digest()
                rng = np.random.default_rng(int.from_bytes(key[:16], 'little'))
                X[i, self.categorical_columns] += rng.random(len(self.categorical_columns))
        return X

    def density(self, X, y):
        """Return log densities including Jacobian and its cancelled log ratio."""
        labels = np.asarray(y)
        if labels.ndim != 1 or len(labels) != len(X) or (not np.isin(labels, self.classes).all()):
            raise ValueError('Intended labels must match fitted binary classes')
        intended = torch.as_tensor((labels == self.classes[1]).astype(np.int64), device=self.device)
        points = self._tensor(X)
        values = torch.empty((len(points), 4), dtype=torch.float64, device=self.device)
        with torch.inference_mode():
            batch_size = self.config.batch_size
            batch = torch.zeros((batch_size, points.shape[1]), dtype=torch.float64, device=self.device)
            batch_labels = torch.zeros(batch_size, dtype=torch.int64, device=self.device)
            rows = torch.arange(batch_size, device=self.device)
            for offset in range(0, len(points), batch_size):
                count = min(batch_size, len(points) - offset)
                batch[:count].copy_(points[offset:offset + count])
                batch_labels[:count].copy_(intended[offset:offset + count])
                if count < batch_size:
                    batch[count:].zero_()
                    batch_labels[count:].zero_()
                z, determinant = self.model(batch)
                gaussian = self.model.gaussian_log_density(z)
                ratio = self.model.gaussian_log_ratio(z, batch_labels)
                destination = values[offset:offset + count]
                destination[:, 0] = (gaussian[rows, batch_labels] + determinant)[:count]
                destination[:, 1] = (gaussian[rows, 1 - batch_labels] + determinant)[:count]
                destination[:, 2] = ratio[:count]
                destination[:, 3] = determinant[:count]
        values = values.cpu().numpy()
        return dict(zip(('log_likelihood_intended', 'log_likelihood_opposing', 'log_ratio', 'log_determinant'), values.T))

    def save(self, path):
        """Checkpoint complete frozen model, private preprocessing and provenance."""
        torch.save({'implementation_version': IMPLEMENTATION_VERSION, 'state_dict': self.model.state_dict(), 'preprocessor': self.preprocessor, 'classes': self.classes, 'fit_row_ids': self.fit_row_ids, 'config': asdict(self.config), 'seed': self.seed, 'audit': self.audit}, path)

    @classmethod
    def load(cls, path, device='cpu'):
        """Load a trusted local checkpoint; sklearn preprocessing uses pickle."""
        checkpoint = torch.load(path, map_location=device, weights_only=False)
        if checkpoint['implementation_version'] != IMPLEMENTATION_VERSION:
            raise ValueError('Unsupported FlowGMM implementation version')
        config = FlowGMMConfig(**checkpoint['config'])
        state = checkpoint['state_dict']
        model = SharedRealNVP(state['means'].shape[1], config, state['means'].cpu().numpy(), state['log_stds'].exp().cpu().numpy()).to(device)
        model.load_state_dict(state)
        return cls(model, checkpoint['preprocessor'], checkpoint['classes'], checkpoint['fit_row_ids'], config, checkpoint['seed'], checkpoint['audit'])

def fit_flow_gmm(frame, y, row_ids, seed, config=None):
    """Supervised class-balanced NLL, fixed epochs, no validation/retries/refits.

    Caller must supply real fit rows only. Calibration, validation, candidates
    and test rows are neither accepted by this API nor used to fit preprocessing.
    """
    config = FlowGMMConfig() if config is None else config
    if not isinstance(config, FlowGMMConfig):
        raise TypeError('config must be FlowGMMConfig')
    if not isinstance(frame, pd.DataFrame) or frame.empty or (not len(frame.columns)):
        raise ValueError('FlowGMM needs a nonempty raw feature DataFrame')
    labels, ids = (np.asarray(y), np.asarray(row_ids))
    if labels.ndim != 1 or ids.ndim != 1 or len(frame) != len(labels) or (len(ids) != len(frame)):
        raise ValueError('Real frame, labels and row IDs must align')
    if len(set(ids.tolist())) != len(ids):
        raise ValueError('Fit row IDs must be unique')
    classes, counts = np.unique(labels, return_counts=True)
    if len(classes) != 2 or pd.isna(labels).any():
        raise ValueError('FlowGMM requires both real binary classes')
    if isinstance(seed, (bool, np.bool_)) or not isinstance(seed, Integral) or (not 0 <= seed < 2 ** 63):
        raise ValueError('seed must be a nonnegative integer below 2**63')
    device = 'cuda' if config.device == 'auto' and torch.cuda.is_available() else config.device
    if device == 'auto':
        device = 'cpu'
    if device == 'cuda' and (not torch.cuda.is_available()):
        raise RuntimeError('Requested CUDA is unavailable')
    start = perf_counter()
    preprocessor = make_preprocessor(frame)
    base = np.asarray(preprocessor.fit_transform(frame), dtype=np.float64)
    if not np.isfinite(base).all():
        raise ValueError('Nonfinite real fitting representation')
    _, _, _, groups = _layout(preprocessor)
    categorical = tuple((column for group in groups for column in group))
    encoded_y = (labels == classes[1]).astype(np.int64)
    rng = np.random.default_rng(seed)
    initial = base.copy()
    initial[:, categorical] += 0.5
    means = np.stack([initial[encoded_y == i].mean(0) for i in range(2)])
    variances = np.stack([base[encoded_y == i].var(0) for i in range(2)])
    variances[:, categorical] += 1 / 12
    stds = np.sqrt(np.maximum(variances, 0.0001))
    cuda_devices = [torch.cuda.current_device()] if device == 'cuda' else []
    with torch.random.fork_rng(devices=cuda_devices):
        torch.random.default_generator.manual_seed(int(seed))
        if cuda_devices:
            torch.cuda.manual_seed(int(seed))
        model = SharedRealNVP(base.shape[1], config, means, stds).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
        targets = torch.as_tensor(encoded_y, device=device)
        weights = torch.as_tensor(len(labels) / (2 * counts[encoded_y]), dtype=torch.float64, device=device)
        losses = []
        model.train()
        for epoch in range(config.epochs):
            jittered = base.copy()
            jittered[:, categorical] += rng.random((len(base), len(categorical)))
            epoch_X = torch.as_tensor(jittered, dtype=torch.float64, device=device)
            order = rng.permutation(len(base))
            total = 0.0
            for offset in range(0, len(base), config.batch_size):
                indices = torch.as_tensor(order[offset:offset + config.batch_size], device=device)
                z, determinant = model(epoch_X[indices])
                log_density = model.gaussian_log_density(z)
                batch_labels = targets[indices]
                own = log_density[torch.arange(len(indices), device=device), batch_labels]
                loss = (-(own + determinant) * weights[indices]).mean()
                if not torch.isfinite(loss):
                    raise RuntimeError(f'Nonfinite FlowGMM training loss at epoch {epoch + 1}')
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                if any((p.grad is not None and (not torch.isfinite(p.grad).all()) for p in model.parameters())):
                    raise RuntimeError(f'Nonfinite FlowGMM training gradient at epoch {epoch + 1}')
                optimizer.step()
                if any((not torch.isfinite(p).all() for p in model.parameters())):
                    raise RuntimeError(f'Nonfinite FlowGMM training parameter at epoch {epoch + 1}')
                std = model.log_stds.detach().exp()
                if not torch.isfinite(std).all() or not (std > 0).all():
                    raise RuntimeError(f'Nonfinite FlowGMM training covariance at epoch {epoch + 1}')
                total += loss.item() * len(indices)
            losses.append(total / len(base))
    audit = {'implementation_version': IMPLEMENTATION_VERSION, 'source_sha256': _source_sha(), 'torch_version': str(torch.__version__), 'reference_commit': REFERENCE_COMMIT, 'reference_paper': 'https://proceedings.mlr.press/v119/izmailov20a.html', 'reference_repository': 'https://github.com/izmailovpavel/flowgmm', 'reference_only': True, 'vendor_code': False, 'model_config': asdict(config), 'resolved_device': device, 'model_checksum': _checksum(model), 'fit_source': 'real_fit_rows_only', 'fit_row_ids': ids.tolist(), 'fit_class_counts': {str(c): int(n) for c, n in zip(classes, counts)}, 'fit_rows': len(base), 'synthetic_training_count': 0, 'equal_priors': [0.5, 0.5], 'training_seconds': perf_counter() - start, 'training_losses': losses, 'objective': 'supervised_class_balanced_negative_log_likelihood', 'early_stopping': False, 'retries': 0, 'frozen': True, 'density_measure': 'dequantized_continuous_proxy_in_detector_coordinates_not_discrete_probability', 'categorical_representation': 'full_onehot_uniform_dequantization', 'train_jitter': 'fresh_each_epoch', 'eval_jitter': 'content_keyed_shared_between_classes', 'numeric_counts': 'continuous_as_U2_interpolation', 'support_filter': False, 'calibration_used': False}
    return FlowGMMCore(model, preprocessor, classes, ids, config, seed, audit)

class FlowGate:
    """Inverse classifier-coordinate bridge and frozen F1 log-space acceptance."""

    def __init__(self, core, classifier_preprocessor, *, calibration_row_ids, validation_row_ids):
        self.core, self.classifier_preprocessor = (core, classifier_preprocessor)
        self.calibration_row_ids = np.asarray(calibration_row_ids).copy()
        self.validation_row_ids = np.asarray(validation_row_ids).copy()
        for ids in (self.calibration_row_ids, self.validation_row_ids):
            if ids.ndim != 1 or len(set(ids.tolist())) != len(ids):
                raise ValueError('Detector role row IDs must be unique one-dimensional vectors')
            ids.setflags(write=False)
        roles = [set(ids.tolist()) for ids in (core.fit_row_ids, self.calibration_row_ids, self.validation_row_ids)]
        if any((roles[a] & roles[b] for a, b in ((0, 1), (0, 2), (1, 2)))):
            raise ValueError('Flow fit/calibration/validation row IDs must be disjoint')
        self.role, self.calibration_class_counts = (None, {})
        self.inference_seconds, self.inference_rows = (0.0, 0)
        self._numeric, self._categorical, self._encoder, self._groups = _layout(classifier_preprocessor)
        self._columns = classifier_preprocessor.named_steps['columns']
        self._width = max((block.stop for block in self._columns.output_indices_.values()))
        self._raw_columns = list(classifier_preprocessor.feature_names_in_)
        if self._raw_columns != list(core.preprocessor.feature_names_in_):
            raise ValueError('Classifier and detector raw feature schemas must match')

    @property
    def audit(self):
        return {**self.core.audit, 'role': self.role, 'calibration_row_ids': self.calibration_row_ids.tolist(), 'validation_row_ids': self.validation_row_ids.tolist(), 'roles_disjoint': True, 'calibration_class_counts': dict(self.calibration_class_counts), 'calibration_used': False, 'support_filter': False, 'inference_seconds': self.inference_seconds, 'inference_rows': self.inference_rows}

    def inverse_classifier(self, X):
        """Decode valid standardized numeric/full-one-hot classifier coordinates."""
        raw = pd.DataFrame(index=np.arange(len(X)), columns=self._raw_columns)
        if self._numeric:
            scaler = self._columns.named_transformers_['numeric'].named_steps['scale']
            numeric_block = self._columns.output_indices_['numeric']
            values = scaler.inverse_transform(X[:, numeric_block])
            for i, name in enumerate(self._numeric):
                raw[name] = values[:, i]
        for name, categories, group in zip(self._categorical, self._encoder.categories_ if self._encoder is not None else (), self._groups):
            raw[name] = categories[np.argmax(X[:, group], axis=1)]
        return raw

    def predict(self, X, y, tau):
        if isinstance(tau, (bool, np.bool_)) or not np.isfinite(tau) or (not 0 < tau < 1):
            raise ValueError('tau must be strictly between 0 and 1')
        points, labels = (np.asarray(X), np.asarray(y))
        if points.ndim != 2 or points.shape[1] != self._width or (not np.issubdtype(points.dtype, np.number)) or np.iscomplexobj(points):
            raise ValueError("Candidate coordinates must have the classifier's numeric feature width")
        if labels.ndim != 1 or len(labels) != len(points) or (not np.isin(labels, self.core.classes).all()):
            raise ValueError('Candidate labels must match the fitted binary classes')
        start = perf_counter()
        n = len(points)
        valid = np.isfinite(points).all(axis=1)
        reasons = np.full(n, 'accepted', dtype=object)
        reasons[~valid] = 'nonfinite_domain'
        for group in self._groups:
            block = points[:, group]
            correct = np.isin(block, (0, 1)).all(axis=1) & (block.sum(axis=1) == 1)
            reasons[valid & ~correct] = 'invalid_onehot'
            valid &= correct
        result = {key: np.full(n, np.nan, dtype=np.float64) for key in ('log_likelihood_intended', 'log_likelihood_opposing', 'log_ratio', 'compatibility_score', 'log_determinant')}
        indices = np.flatnonzero(valid)
        if len(indices):
            raw = self.inverse_classifier(points[indices].astype(np.float64))
            detector_X = np.asarray(self.core.preprocessor.transform(raw), dtype=np.float64)
            finite_representation = np.isfinite(detector_X).all(axis=1)
            if self._numeric:
                finite_representation &= np.isfinite(raw[self._numeric].to_numpy(dtype=np.float64)).all(axis=1)
            reasons[indices[~finite_representation]] = 'nonfinite_score'
            valid[indices[~finite_representation]] = False
            known = finite_representation.copy()
            for group in self.core.categorical_groups:
                known &= detector_X[:, group].sum(axis=1) == 1
            reasons[indices[finite_representation & ~known]] = 'unknown_flow_category'
            valid[indices[~known]] = False
            indices = indices[known]
            if len(indices):
                represented = self.core._dequantize(detector_X if known.all() else detector_X[known])
                scores = self.core.density(represented, labels[indices])
                finite = np.ones(len(indices), dtype=bool)
                for key, values in scores.items():
                    result[key][indices] = values
                    finite &= np.isfinite(values)
                reasons[indices[~finite]] = 'nonfinite_score'
                valid[indices[~finite]] = False
                ratio = scores['log_ratio']
                result['compatibility_score'][indices] = np.exp(-np.logaddexp(0.0, -ratio))
        accepted = valid & (result['log_ratio'] >= math.log(tau) - math.log1p(-tau))
        reasons[valid & ~accepted] = 'flow_incompatible'
        self.inference_seconds += perf_counter() - start
        self.inference_rows += n
        return {**result, 'valid': valid, 'accepted': accepted, 'reasons': reasons}
'''


In [ ]:
# Bundled module: u2.py
BUNDLE_SOURCES['u2.py'] = r'''"""Real-only canonical U2, with native relocation or a frozen FlowGMM gate.

Grouping, quotas, MMA scoring, interpolation, relocation and stopping are reused
from the bundled core. The gate replaces both post-neighbor detection and
relocation; rejected attempts consume quotas and are never refilled. Candidate
records contain ancestry and gate diagnostics, not feature-coordinate copies.
"""
from dataclasses import asdict, fields

import numpy as np
import torch

from portable_ugo.allocation import AllocationConfig, GroupAllocator
from portable_ugo.groups import GroupConfig, prepare_groups
from portable_ugo.ugo import (
    Candidates, GroupBudget, RealSupport, UGOConfig, UGOPipeline,
    accept_relocation, nearest_indices, noise_pre_detect, propose_interpolation,
)


def _numpy(value):
    return value.detach().cpu().numpy() if isinstance(value, torch.Tensor) else np.asarray(value)


def _configurations(parameters):
    values = dict(parameters)
    tau = values.pop("f1_tau", .5)
    group_names = {item.name for item in fields(GroupConfig)}
    allocation_names = {item.name for item in fields(AllocationConfig)}
    grouping = GroupConfig(**{name: values.pop(name) for name in group_names if name in values})
    allocation = AllocationConfig(**{name: values.pop(name) for name in allocation_names if name in values})
    if grouping != GroupConfig():
        raise ValueError("Canonical U2 requires KMeans-8 and five 80% real-row subsamples")
    if allocation != AllocationConfig(group_alpha=allocation.group_alpha):
        raise ValueError("Canonical U2 requires stability allocation, group cap 2n and seed cap 4")
    if values.get("lambda_select", .5) != .5:
        raise ValueError("Canonical U2 uses the top half of native MMA-ranked real seeds")
    return grouping, allocation, tau, values


def prepare_u2(X, y, seed, targets, parameters):
    """Prepare reusable real-only groups; alpha and gate tau do not affect them.

    ``targets`` contains final class counts, not synthetic counts. With an
    explicit partial mapping, unspecified classes retain their real counts.
    Without targets, every observed class targets the largest real class count.
    """
    grouping, _, _, _ = _configurations(parameters)
    real_X, real_y = _numpy(X), _numpy(y)
    # Reuse core validation and normalization even before group memberships exist.
    target_counts = GroupAllocator({"groups": []}, real_y, targets).target_counts
    labels = [label for label in target_counts
              if target_counts[label] > int(np.count_nonzero(real_y == label))]
    return prepare_groups(real_X, real_y, labels, seed, grouping)


def run_u2(X, y, model_fn, *, seed, targets, parameters, max_iterations, device,
           categorical_columns=(), prepared=None, flow_support=None):
    """Return the core UGOResult for native U2 or U2 plus a frozen FlowGate.

    ``model_fn(X_tensor, y_tensor)`` fits the actual downstream learner and must
    expose probabilities in consecutive class-label order. Native U2 is used
    for learner/alpha selection; supplying an already-fitted ``flow_support``
    invokes only its ``predict(X, y, tau)`` and never fits or calibrates it.
    Parent indices always reference the original real training matrix. All
    categorical output columns are copied together from the real partner before
    either post-handler. Accepted coordinates remain available in result.X;
    candidate_log deliberately does not duplicate proposed or final coordinates.
    """
    grouping, allocation, tau, base = _configurations(parameters)
    if flow_support is not None and (isinstance(tau, (bool, np.bool_))
            or not np.isfinite(tau) or not 0 < tau < 1):
        raise ValueError("f1_tau must be strictly between zero and one")
    base.update(random_state=seed, categorical_columns=tuple(categorical_columns),
                max_iterations=max_iterations, device=device)
    config = UGOConfig(**base)
    real_y = _numpy(y)
    prepared = (prepare_u2(X, real_y, seed, targets, parameters)
                if prepared is None else prepared)
    if prepared["configuration"] != asdict(grouping):
        raise ValueError("Prepared groups do not match canonical U2 grouping")
    allocator = GroupAllocator(prepared, real_y, targets, allocation)
    target_labels = {label for label in allocator.target_counts
                     if allocator.target_counts[label] > allocator.real_counts[label]}
    proposal_cap = sum(allocator.target_counts[label] - allocator.real_counts[label]
                       for label in allocator.target_counts)
    supported = [group for group in prepared["groups"]
                 if group["real_support"] >= 2 and group["distinct_coordinates"] >= 2
                 and group["stability"] > 0]
    supported_ids = {group["id"] for group in supported}
    protected_indices = (np.concatenate([group["indices"] for group in supported])
                         if supported else np.empty(0, dtype=np.int64))
    accepted_seed = np.zeros(len(real_y), dtype=np.int64)
    candidate_log = []
    conflict = None
    proposal_round = 0

    def support(features, labels, cfg):
        nonlocal conflict
        classes = torch.unique(labels, sorted=True)
        conflict = noise_pre_detect(features, labels, cfg.k_pre, cfg.distance_batch_size)
        eligible = ~conflict
        # Cross-class conflict is diagnostic, not a minority corruption label:
        # supported minority rows override pre-noise exclusion, unsupported ones
        # cannot become parents even when their local neighbors agree.
        for label in target_labels:
            eligible[labels == label] = False
        eligible[torch.as_tensor(protected_indices, device=features.device)] = True
        partners, indices_by_class, excluded = {}, {}, []
        for label in classes.tolist():
            indices = torch.nonzero((labels == label) & eligible, as_tuple=True)[0]
            if len(indices) < 2 or len(torch.unique(features[indices], dim=0)) < 2:
                eligible[indices] = False
                excluded.append(label)
                continue
            indices_by_class[label] = indices
            neighbors = nearest_indices(
                features[indices], features[indices], min(cfg.k_pre, len(indices) - 1),
                cfg.distance_batch_size, torch.arange(len(indices), device=features.device),
                exclude_zero=True,
            )
            partners[label] = indices[neighbors]
        return RealSupport(features, labels, classes, eligible, partners, indices_by_class, excluded)

    def budget(real, scores, remaining, cfg):
        available = [label for label in real.group_indices if label in target_labels
                     and allocator.used_class[label]
                     < allocator.target_counts[label] - allocator.real_counts[label]]
        # In gated runs generated counts only accepted rows. Bound by attempts
        # separately so rejection cannot turn fixed quotas into refill budgets.
        remaining = min(remaining, proposal_cap - int(allocator.used_seed.sum()))
        if remaining <= 0:
            return None
        count = min(remaining, max(1, round(len(real.y) * cfg.lambda_step)))
        score_values = scores.detach().cpu().numpy()
        for target in sorted(available,
                             key=lambda label: float(scores[real.group_indices[label]].mean()),
                             reverse=True):
            plan = allocator.allocate(score_values, target, count, cfg.lambda_select)
            if plan is not None:
                seeds = torch.as_tensor(plan["seed_indices"], device=real.X.device, dtype=torch.long)
                return GroupBudget(target, seeds, plan["count"])
        return None

    def propose(real, planned, rng, cfg):
        return propose_interpolation(real, planned, rng, cfg, seed_indices=planned.seed_indices)

    def accept(real, candidates, cfg):
        nonlocal proposal_round
        proposal_round += 1
        # Commit actual proposals BEFORE the gate, never just accepted rows.
        # Distinct real-coordinate support ensures interpolation has a partner.
        seeds = candidates.seed_indices.detach().cpu().numpy()
        allocator.commit(seeds)
        scores = None
        if flow_support is None:
            accepted = accept_relocation(real, candidates, cfg)
            keep = np.ones(len(seeds), dtype=bool)
        else:
            accepted = candidates
            scores = flow_support.predict(candidates.X.detach().cpu().numpy(),
                                          candidates.y.detach().cpu().numpy(), tau)
            keep = np.asarray(scores["accepted"], dtype=bool)
            if keep.shape != seeds.shape:
                raise ValueError("FlowGate must return one acceptance decision per proposal")
        np.add.at(accepted_seed, seeds[keep], 1)
        if allocator.records:
            allocator.records[-1].update(attempted=len(seeds), accepted=int(keep.sum()))
        partners = accepted.partner_indices.detach().cpu().numpy()
        relocations = accepted.relocation_indices.detach().cpu().numpy()
        labels = accepted.y.detach().cpu().numpy()
        for index, source in enumerate(seeds):
            entry = {"proposal_id": len(candidate_log), "round": proposal_round,
                     "seed_index": int(source), "partner_index": int(partners[index]),
                     "relocation_index": int(relocations[index]),
                     "category_donor_index": int(relocations[index] if relocations[index] >= 0
                                                 else partners[index]),
                     "group_id": int(prepared["row_group_ids"][source]),
                     "class": int(labels[index]), "accepted": bool(keep[index]),
                     "reason": "accepted" if scores is None else str(scores["reasons"][index])}
            if scores is not None:
                entry.update(threshold=float(tau),
                             log_ratio_threshold=float(np.log(tau) - np.log1p(-tau)))
                for key in ("log_likelihood_intended", "log_likelihood_opposing", "log_ratio",
                            "compatibility_score", "log_determinant"):
                    value = float(scores[key][index])
                    entry[key] = value if np.isfinite(value) else None
            candidate_log.append(entry)
        if flow_support is not None:
            mask = torch.as_tensor(keep, device=accepted.X.device)
            accepted = Candidates(*(getattr(accepted, key)[mask] for key in
                                    ("X", "y", "seed_indices", "partner_indices", "relocation_indices")))
        return accepted

    result = UGOPipeline(config, support=support, group_budget=budget,
                         propose=propose, accept=accept).run(
                             torch.as_tensor(X), torch.as_tensor(y), model_fn)
    conflict_numpy = conflict.detach().cpu().numpy()
    actual_counts = {label: int((result.y == label).sum()) for label in allocator.real_counts}
    groups = []
    for group in prepared["groups"]:
        indices = group["indices"]
        groups.append({**group, "supported": group["id"] in supported_ids,
                       "cross_class_conflict_count": int(conflict_numpy[indices].sum()),
                       "attempted_generated": int(allocator.used_seed[indices].sum()),
                       "actual_generated": int(accepted_seed[indices].sum())})
    # Core traces include a per-group per-round seed dictionary. Final attempted
    # and accepted multiplicities already retain that information compactly.
    records = [{**record, "group_trace": [
        {key: value for key, value in trace.items() if key != "used_seed"}
        for trace in record["group_trace"]]} for record in allocator.records]
    attempted = int(allocator.used_seed.sum())
    accepted_count = int(accepted_seed.sum())
    result.support_summary.update(
        target_counts=allocator.target_counts, actual_counts=actual_counts,
        protocol_deviations=["matched_per_class_target_caps"] if targets is not None else [],
        group_support={
            "configuration": prepared["configuration"], "allocation": asdict(allocation),
            "row_group_ids": prepared["row_group_ids"], "groups": groups,
            "real_row_ids": prepared.get("real_row_ids"), "synthetic_training_count": 0,
            "cross_class_conflict_indices": np.flatnonzero(conflict_numpy),
            "protected_conflict_indices": protected_indices[conflict_numpy[protected_indices]],
            "seed_multiplicity": {str(i): int(accepted_seed[i]) for i in np.flatnonzero(accepted_seed)},
            "attempted_seed_multiplicity": {
                str(i): int(allocator.used_seed[i]) for i in np.flatnonzero(allocator.used_seed)},
            "allocation_records": records,
            "partner_scope": "eligible_same_class_real_neighbors_not_group_restricted",
        },
        candidate_log=candidate_log,
        proposal_budget={
            "cap": proposal_cap, "attempted": attempted, "accepted": accepted_count,
            "rejected": attempted - accepted_count, "unspent_proposals": proposal_cap - attempted,
            "allocation_accounting": "planned proposals; rejected quotas never refilled",
        },
        post_handler="native_neighbor_relocation" if flow_support is None else "frozen_flow_gate",
        categorical_policy="copy_real_partner_before_post_handler; native_relocation_copies_real_donor_row",
        synthetic_sample_weight=1.,
    )
    accepted_proposals = (entry for entry in candidate_log if entry["accepted"])
    for ancestry, proposal in zip(result.provenance, accepted_proposals):
        ancestry.update(group_id=proposal["group_id"], proposal_id=proposal["proposal_id"],
                        category_donor_index=proposal["category_donor_index"])
    if flow_support is not None:
        result.support_summary["flow_support"] = flow_support.audit
        if result.stop_reason == "no_valid_partner" and candidate_log:
            result.stop_reason = "f1_no_candidates_accepted"
    if result.stop_reason == "no_eligible_group" and attempted >= proposal_cap:
        result.stop_reason = "proposal_cap_reached"
    if all(actual_counts[label] == allocator.target_counts[label] for label in actual_counts):
        result.stop_reason = "target_counts_reached"
    # Keep the terminal round consistent when the wrapper gives a more precise
    # quota/gate explanation than the generic core pipeline.
    if result.rounds and "stop_reason" in result.rounds[-1]:
        result.rounds[-1]["stop_reason"] = result.stop_reason
    return result
'''


In [ ]:
# Bundled module: models.py
BUNDLE_SOURCES['models.py'] = r'''"""Native downstream learners; no experiment/repository runtime dependencies."""
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import numpy as np
import torch
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import ParameterSampler
from sklearn.svm import SVC


CLASSIFIERS = ("mlp", "logistic_regression", "xgboost", "random_forest", "svm")
_SPACES = {
    "logistic_regression": {"C": [.01, .1, 1., 10., 100.]},
    "random_forest": {"n_estimators": [100, 200, 400], "max_depth": [None, 6, 12],
                      "min_samples_leaf": [1, 3, 8], "max_features": ["sqrt", .8]},
    "xgboost": {"n_estimators": [100, 200, 400], "max_depth": [3, 6, 9],
                "learning_rate": [.01, .05, .1], "subsample": [.7, 1.],
                "colsample_bytree": [.7, 1.]},
    "svm": {"C": [.1, 1., 10., 100.], "gamma": ["scale", .001, .01, .1]},
    "mlp": {"hidden_dim": [20, 64, 128], "lr": [.0003, .001, .003]},
}


def as_array(value):
    return value.detach().cpu().numpy() if isinstance(value, torch.Tensor) else np.asarray(value)


def resolve_device(device):
    if device not in ("auto", "cpu", "cuda"):
        raise ValueError("DEVICE must be 'auto', 'cpu', or 'cuda'")
    if device == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA explicitly requested but unavailable; no CPU substitution")
    return "cuda" if device != "cpu" and torch.cuda.is_available() else "cpu"


class MLPClassifierAdapter:
    """Root two-hidden-layer ReLU/dropout(.25), full-batch RMSprop, fixed epochs.

    Prediction is chunked, including native MMA calls, to bound temporary VRAM.
    There is deliberately no validation argument or early stopping.
    """
    def __init__(self, *, hidden_dim=20, lr=.001, epochs=160, random_state=42,
                 device="cpu", prediction_batch_size=4096):
        self.hidden_dim, self.lr, self.epochs = hidden_dim, lr, epochs
        self.random_state, self.device = random_state, resolve_device(device)
        self.prediction_batch_size = prediction_batch_size
        self.backend_ = f"pytorch_{self.device}"

    def fit(self, X, y):
        torch.manual_seed(self.random_state)
        torch.use_deterministic_algorithms(True)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        features = torch.as_tensor(X, dtype=torch.float32, device=self.device)
        labels = torch.as_tensor(y, dtype=torch.long, device=self.device)
        self.classes_ = np.unique(as_array(labels))
        if not np.array_equal(self.classes_, np.arange(len(self.classes_))):
            raise ValueError("MLP labels must be consecutive integers starting at zero")
        self.model_ = torch.nn.Sequential(
            torch.nn.Linear(features.shape[1], self.hidden_dim), torch.nn.ReLU(),
            torch.nn.Dropout(.25), torch.nn.Linear(self.hidden_dim, self.hidden_dim),
            torch.nn.ReLU(), torch.nn.Dropout(.25),
            torch.nn.Linear(self.hidden_dim, len(self.classes_))).to(self.device)
        optimizer = torch.optim.RMSprop(self.model_.parameters(), lr=self.lr)
        self.model_.train()
        for epoch in range(self.epochs):
            optimizer.zero_grad(set_to_none=True)
            loss = torch.nn.functional.cross_entropy(self.model_(features), labels)
            if not torch.isfinite(loss):
                raise RuntimeError(f"Nonfinite MLP loss at epoch {epoch + 1}")
            loss.backward()
            optimizer.step()
        self.model_.eval()
        self.best_epoch_ = self.epochs
        self.training_count_ = len(labels)
        return self

    def predict_proba_tensor(self, X):
        chunks = []
        destination = X.device if isinstance(X, torch.Tensor) else torch.device("cpu")
        self.model_.eval()
        with torch.inference_mode():
            for start in range(0, len(X), self.prediction_batch_size):
                features = torch.as_tensor(X[start:start + self.prediction_batch_size],
                                           dtype=torch.float32, device=self.device)
                chunks.append(torch.softmax(self.model_(features), dim=1).to(destination))
        return (torch.cat(chunks) if chunks else
                torch.empty((0, len(self.classes_)), device=destination))

    def predict_proba(self, X):
        return self.predict_proba_tensor(X).cpu().numpy()

    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(axis=1)]


def candidate_parameters(name, seed, budget, smoke=False):
    """Exactly the declared number, default first; finite spaces sample with replacement."""
    if name not in CLASSIFIERS:
        raise ValueError(f"Unknown classifier: {name}")
    rng = np.random.RandomState(seed)
    candidates = [{}]
    for _ in range(budget - 1):
        candidates.append(next(iter(ParameterSampler(_SPACES[name], n_iter=1, random_state=rng))))
    if smoke and name in ("random_forest", "xgboost"):
        candidates = [{**item, "n_estimators": 12} for item in candidates]
    return candidates


def make_estimator(name, seed, parameters, *, device, epochs, n_jobs):
    """CPU sklearn controls and genuinely selected Torch/XGBoost CPU or CUDA."""
    selected = resolve_device(device)
    if name == "mlp":
        return MLPClassifierAdapter(epochs=epochs, random_state=seed, device=selected, **parameters)
    if name == "logistic_regression":
        model = LogisticRegression(max_iter=1000, solver="lbfgs", random_state=seed, **parameters)
        backend = "sklearn_cpu"
    elif name == "random_forest":
        values = dict(n_estimators=200, random_state=seed, n_jobs=n_jobs)
        model = RandomForestClassifier(**{**values, **parameters})
        backend = "sklearn_cpu"
    elif name == "svm":
        model = SVC(kernel="rbf", probability=True, random_state=seed, **parameters)
        backend = "sklearn_cpu"
    elif name == "xgboost":
        from xgboost import XGBClassifier
        values = dict(n_estimators=200, max_depth=6, learning_rate=.05, subsample=.8,
                      colsample_bytree=.8, eval_metric="logloss", tree_method="hist",
                      device=selected, random_state=seed, n_jobs=n_jobs)
        model = XGBClassifier(**{**values, **parameters})
        backend = f"xgboost_{selected}"
    else:
        raise ValueError(f"Unknown classifier: {name}")
    model.backend_ = backend
    return model


def fit_estimator(model, X, y):
    if isinstance(model, MLPClassifierAdapter):
        return model.fit(X, y)
    model.fit(as_array(X), as_array(y))
    if model.backend_ == "xgboost_cuda":
        import json
        actual = json.loads(model.get_booster().save_config())["learner"]["generic_param"]["device"]
        if not str(actual).startswith("cuda"):
            raise RuntimeError(f"XGBoost requested CUDA but fitted on {actual}; refusing silent fallback")
    return model
'''


In [ ]:
# Bundled module: sampling.py
BUNDLE_SOURCES['sampling.py'] = r'''"""Training-role-only SMOTE, KMeansSMOTE and random walk oversampling.

SMOTE and KMeansSMOTE are the actual imbalanced-learn CPU algorithms on
numeric plus full one-hot coordinates, NOT SMOTENC. Only synthetic rows have
each complete categorical block projected to its argmax (first category breaks
ties). No validation/test observations or synthetic feedback fit any sampler.
KMeansSMOTE's ceil-rounded excess is deterministically trimmed, never refilled.

RWO is independently implemented from the numerical convention described at
https://raw.githubusercontent.com/analyticalmindsltd/smote_variants/master/smote_variants/oversampling/_rwo_sampling.py
and the nominal roulette convention described at
https://raw.githubusercontent.com/cran/imbalance/master/R/rwo.R .
For each class, a uniform real seed receives independent numeric noise with
scale std(real class, ddof=0) / sqrt(real class size). Each nominal variable is
sampled independently from real-class category frequencies. The R reference
uses a variance variant: this implementation follows the Python reference's
standard-deviation convention, not a bitwise reproduction of the R code.
There is no walk chain, synthetic-statistics update, numeric clipping, integer
rounding, ROS substitution, or copied third-party implementation code.
"""
from collections.abc import Mapping
from numbers import Integral, Real

import numpy as np


SAMPLER_DEFAULTS = {
    "smote": {"k_neighbors": 5},
    "kmeans_smote": {"k_neighbors": 2, "kmeans_estimator": 8,
                     "cluster_balance_threshold": 0.0, "density_exponent": "auto"},
    "rwo": {},
}


def _integer(value, name, minimum=1):
    if isinstance(value, (bool, np.bool_)) or not isinstance(value, Integral) or value < minimum:
        raise ValueError(f"{name} must be an integer >= {minimum}")
    return int(value)


def _method_parameters(method, parameters):
    if method not in SAMPLER_DEFAULTS:
        raise ValueError(f"Unknown sampling control: {method!r}")
    if parameters is None:
        parameters = {}
    if not isinstance(parameters, Mapping):
        raise ValueError(f"{method} parameters must be a mapping")
    unknown = set(parameters) - set(SAMPLER_DEFAULTS[method])
    if unknown:
        raise ValueError(f"Unsupported {method} parameters: {sorted(unknown, key=str)}; "
                         "use target_counts and seed for the sampling budget/random state")
    values = {**SAMPLER_DEFAULTS[method], **parameters}
    for name in ("k_neighbors", "kmeans_estimator"):
        if name in values:
            values[name] = _integer(values[name], f"{method}.{name}")
    for name in ("cluster_balance_threshold", "density_exponent"):
        if name not in values:
            continue
        value = values[name]
        if isinstance(value, str) and value == "auto":
            continue
        if (isinstance(value, (bool, np.bool_)) or not isinstance(value, Real)
                or not np.isfinite(value) or value < 0
                or (name == "cluster_balance_threshold" and value > 1)):
            interval = "[0, 1]" if name == "cluster_balance_threshold" else "[0, infinity)"
            raise ValueError(f"{method}.{name} must be 'auto' or a finite number in {interval}")
        values[name] = float(value)
    return values


def validate_sampler_parameters(parameters=None):
    """Return independent JSON-safe defaults/options for exactly the three controls.

    Supplied top-level mappings must contain all three method keys. Within each
    method, omitted known options take their prespecified defaults. Estimator
    objects and competing random_state/sampling_strategy options are unsupported.
    The fixed inclusive KMeansSMOTE threshold 0.0 permits rare-positive clusters;
    it is not a per-dataset fallback or an extra searched configuration.
    """
    if parameters is None:
        parameters = SAMPLER_DEFAULTS
    if not isinstance(parameters, Mapping) or set(parameters) != set(SAMPLER_DEFAULTS):
        raise ValueError("SAMPLER_PARAMETERS must have exactly smote, kmeans_smote and rwo keys")
    return {method: _method_parameters(method, parameters[method]) for method in SAMPLER_DEFAULTS}


def _counts(y):
    labels, counts = np.unique(y, return_counts=True)
    return {int(label): int(count) for label, count in zip(labels, counts)}


def _targets(y, target_counts):
    original = _counts(y)
    if not isinstance(target_counts, Mapping):
        raise ValueError("target_counts must map class labels to final integer counts")
    supplied = {}
    for label, count in target_counts.items():
        label = _integer(label, "target_counts class label", minimum=np.iinfo(np.int64).min)
        if label not in original:
            raise ValueError("target_counts cannot introduce an absent real class")
        supplied[label] = _integer(count, "target_counts final count", minimum=0)
    targets = {label: supplied.get(label, count) for label, count in original.items()}
    if any(targets[label] < count for label, count in original.items()):
        raise ValueError("target_counts cannot remove real observations")
    return original, targets


def _categorical_groups(X, categorical_groups):
    groups, occupied = [], set()
    for group in categorical_groups:
        if isinstance(group, Integral):
            raise ValueError("categorical_groups requires complete one-hot blocks, not flat indices")
        columns = tuple(_integer(column, "categorical group index", minimum=0) for column in group)
        if not columns or len(set(columns)) != len(columns):
            raise ValueError("Categorical blocks must be nonempty and contain distinct indices")
        if any(column >= X.shape[1] or column in occupied for column in columns):
            raise ValueError("Categorical blocks must be disjoint and within the feature matrix")
        block = X[:, columns]
        if not np.isin(block, (0, 1)).all() or not np.all(block.sum(axis=1) == 1):
            raise ValueError("Real training categorical blocks must be valid full one-hot encodings")
        occupied.update(columns)
        groups.append(columns)
    numeric = tuple(column for column in range(X.shape[1]) if column not in occupied)
    return tuple(groups), numeric


def _project_categories(synthetic, groups):
    for group in groups:
        categories = np.argmax(synthetic[:, group], axis=1)
        synthetic[:, group] = 0
        synthetic[np.arange(len(synthetic)), np.asarray(group)[categories]] = 1


def _rwo(X, y, original, targets, groups, numeric, seed):
    rng = np.random.default_rng(seed)
    count = sum(targets[label] - original[label] for label in original)
    synthetic = np.empty((count, X.shape[1]), dtype=X.dtype)
    labels = np.empty(count, dtype=y.dtype)
    offset = 0
    for label, real_count in original.items():
        needed = targets[label] - real_count
        if not needed:
            continue
        real = X[y == label]
        block = synthetic[offset:offset + needed]
        block[:] = real[rng.integers(real_count, size=needed)]
        if numeric:
            coordinates = real[:, numeric].astype(np.float64, copy=False)
            scale = np.std(coordinates, axis=0, ddof=0) / np.sqrt(real_count)
            # Constant columns stay constant even if floating-point reduction
            # would otherwise introduce a tiny nonzero standard deviation.
            scale[np.all(coordinates == coordinates[0], axis=0)] = 0
            block[:, numeric] += rng.normal(size=(needed, len(numeric))) * scale
        for group in groups:
            frequencies = real[:, group].sum(axis=0, dtype=np.float64) / real_count
            categories = rng.choice(len(group), size=needed, p=frequencies)
            block[:, group] = 0
            block[np.arange(needed), np.asarray(group)[categories]] = 1
        labels[offset:offset + needed] = label
        offset += needed
    return synthetic, labels


def resample_control(method, X, y, target_counts, seed, *, categorical_groups=(),
                     n_jobs=4, parameters=None):
    """Return (X_resampled, y_resampled, compact_audit), with unit row weights.

    Inputs are floating-point classifier-preprocessed REAL training observations
    only; target_counts contains final class counts, not added counts. Original
    feature/label bytes, dtype and order are retained, and inputs are not mutated.
    Zero additional demand returns the original arrays without importing/fitting
    imbalanced-learn. Infeasible neighbors/clusters raise without parameter
    weakening, fallback, hidden duplication or synthetic padding. The caller's
    existing failure logger should capture such failures.
    """
    values = _method_parameters(method, parameters)
    seed = _integer(seed, "seed", minimum=0)
    if seed >= 2 ** 32:
        raise ValueError("seed must be less than 2**32 for imbalanced-learn")
    if isinstance(n_jobs, (bool, np.bool_)) or not isinstance(n_jobs, Integral) or n_jobs == 0 or n_jobs < -1:
        raise ValueError("n_jobs must be -1 or a positive integer")
    X, y = np.asarray(X), np.asarray(y)
    if (X.ndim != 2 or y.ndim != 1 or len(X) != len(y) or not len(y)
            or not X.shape[1] or not np.issubdtype(X.dtype, np.floating)
            or not np.isfinite(X).all() or not np.issubdtype(y.dtype, np.integer)):
        raise ValueError("Controls require aligned nonempty finite floating-point features and integer class labels")
    original, targets = _targets(y, target_counts)
    groups, numeric = _categorical_groups(X, categorical_groups)
    requested = sum(targets[label] - original[label] for label in original)
    implementations = {"smote": "imblearn.over_sampling.SMOTE",
                       "kmeans_smote": "imblearn.over_sampling.KMeansSMOTE",
                       "rwo": "independent_real_class_RWO_std_ddof0"}
    adapter = ("real_class_nominal_frequency_roulette" if method == "rwo" else
               "numeric_plus_full_onehot_then_categorical_argmax_projection") if groups else "numeric_native"
    audit = {"method": method, "implementation": implementations[method],
             "accepted_count": 0, "requested_count": requested, "underspend": requested,
             "real_count": len(y), "real_counts": original, "actual_counts": dict(original),
             "stop_reason": "target_counts_reached", "synthetic_weights": 1,
             "parameters": values, "categorical_adapter": adapter,
             "raw_generated_count": 0, "trimmed_count": 0}
    if not requested:
        return X, y, audit
    if method == "rwo":
        synthetic, labels = _rwo(X, y, original, targets, groups, numeric, seed)
    else:
        from imblearn.over_sampling import KMeansSMOTE, SMOTE

        strategy = {label: count for label, count in targets.items() if count > original[label]}
        options = {**values, "sampling_strategy": strategy, "random_state": seed}
        if method == "kmeans_smote":
            sampler = KMeansSMOTE(n_jobs=int(n_jobs), **options)
        else:
            sampler = SMOTE(**options)
        # Keep the caller's arrays untouched even across dependency versions.
        sampled_X, sampled_y = sampler.fit_resample(X.copy(), y.copy())
        if (sampled_X.ndim != 2 or sampled_X.shape[1] != X.shape[1]
                or sampled_y.ndim != 1 or len(sampled_X) != len(sampled_y)
                or len(sampled_y) < len(y)
                or not np.array_equal(sampled_X[:len(X)], X)
                or not np.array_equal(sampled_y[:len(y)], y)):
            raise ValueError("imbalanced-learn changed real rows/labels or returned misaligned output")
        synthetic, labels = sampled_X[len(X):], sampled_y[len(y):]
    raw_count = len(labels)
    if not np.isfinite(synthetic).all():
        raise ValueError(f"{method} generated nonfinite features")
    if set(_counts(labels)) - set(original):
        raise ValueError(f"{method} generated labels absent from the real training role")
    keep = np.ones(raw_count, dtype=bool)
    for label, real_count in original.items():
        indices = np.flatnonzero(labels == label)
        allowance = targets[label] - real_count
        if len(indices) > allowance:
            if method != "kmeans_smote":
                raise ValueError(f"{method} exceeded the requested class target")
            # Stable prefix selection trims only generated ceil excess; original
            # rows never enter this selection. No attempt fills any shortfall.
            keep[indices[allowance:]] = False
    if not keep.all():
        synthetic, labels = synthetic[keep], labels[keep]
    # Own the synthetic buffer before projection; original rows are never decoded.
    synthetic = np.array(synthetic, dtype=X.dtype, copy=True)
    if method != "rwo" and groups:
        _project_categories(synthetic, groups)
    if not np.isfinite(synthetic).all():
        raise ValueError(f"{method} generated features outside the finite input dtype domain")
    X_res = np.concatenate((X, synthetic), axis=0)
    y_res = np.concatenate((y, labels.astype(y.dtype, copy=False)))
    actual = _counts(y_res)
    accepted = len(labels)
    audit.update(accepted_count=accepted, underspend=requested - accepted,
                 actual_counts=actual, raw_generated_count=raw_count,
                 trimmed_count=raw_count - accepted,
                 stop_reason="target_counts_reached" if accepted == requested else "sampler_underspend")
    return X_res, y_res, audit
'''


In [ ]:
# Bundled module: benchmark.py
BUNDLE_SOURCES['benchmark.py'] = r'''"""Raw-OpenML, F1-first six-method benchmark embedded in the portable notebook.

Scientific and training-only smoke runs are distinctly marked. No test observation
participates in preprocessing, proposal scoring, learner selection or thresholding.
Sklearn LR/RF/RBF-SVM are explicit CPU controls; Torch/Flow/XGBoost obey DEVICE.
"""
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import json
from numbers import Integral
from pathlib import Path
import platform
import re
import sys
from time import perf_counter
import traceback
import uuid

import numpy as np
import pandas as pd
from sklearn.metrics import (average_precision_score, balanced_accuracy_score,
    confusion_matrix, f1_score, matthews_corrcoef, precision_score, recall_score)
from sklearn.model_selection import train_test_split
from threadpoolctl import threadpool_limits
import torch

from .models import (CLASSIFIERS, as_array, candidate_parameters, fit_estimator,
                     make_estimator, resolve_device)
from .preprocessing import categorical_columns, make_preprocessor
from .ugo_original import OriginalUGOConfig, original_ugo
from .u2 import prepare_u2, run_u2
from .flow_gmm import FlowGMMConfig, FlowGate, fit_flow_gmm
from .sampling import SAMPLER_DEFAULTS, resample_control, validate_sampler_parameters

METHODS = ("baseline", "original_ugo", "u2_flow_gate", "smote", "kmeans_smote", "rwo")
SAMPLER_PARAMETERS = SAMPLER_DEFAULTS
DEFAULTS = dict(DATASET_RANGE="1-32", EXCLUDED_DATASETS=(10, 26),
    CLASSIFIERS=list(CLASSIFIERS), SEEDS=[42, 43, 44],
    N_CANDIDATES=10, TEST_FRACTION=.2, VALIDATION_FRACTION=.2,
    AUGMENTATION_CAP=1000, MAX_ROUNDS=20, MLP_EPOCHS=160, FLOW_EPOCHS=60,
    GROUP_ALPHAS=[0., .5, 1.], GATE_TAUS=[.5, .6, .7, .8], DEVICE="auto",
    OUTPUT_DIR="ugo_results", DATA_HOME="ugo_data", N_JOBS=4, SMOKE=False,
    MAX_ROWS=None, SAMPLER_PARAMETERS=SAMPLER_PARAMETERS)
METRICS = ("minority_precision", "minority_recall", "minority_f1", "average_precision",
           "balanced_accuracy", "mcc", "f1_at_0_5", "accepted_count", "requested_count",
           "runtime_seconds", "classifier_fit_count")


def _json_default(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f"Cannot serialize {type(value).__name__}")


def _hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=_json_default,
                                    allow_nan=False).encode()).hexdigest()


def _dump(path, value):
    temporary = path.with_suffix(path.suffix + ".part")
    temporary.write_text(json.dumps(value, indent=2, sort_keys=True,
        default=_json_default, allow_nan=False) + "\n", encoding="utf-8")
    temporary.replace(path)


def _catalogue(catalogue):
    rows = [dict(row) for row in catalogue]
    if not rows:
        raise ValueError("The dataset catalogue is empty")
    for index, row in enumerate(rows, 1):
        if row.get("number") != index or isinstance(row.get("number"), bool):
            raise ValueError("Catalogue numbers must be consecutive and 1-based")
        if not isinstance(row.get("name"), str) or not row["name"]:
            raise ValueError("Each catalogue entry requires a dataset name")
        if isinstance(row.get("openml_id"), bool) or not isinstance(row.get("openml_id"), Integral):
            raise ValueError("Each catalogue entry requires an integer OpenML source ID")
        if "imbalance_type" not in row:
            raise ValueError("Each catalogue entry requires its imbalance type")
    return rows


def show_catalogue(catalogue):
    """Display the numbered raw OpenML sources, without downloading data."""
    return pd.DataFrame(_catalogue(catalogue))[["number", "openml_id", "name", "imbalance_type"]]


def _excluded_dataset_numbers(numbers):
    if not isinstance(numbers, (list, tuple)) or any(
            isinstance(number, (bool, np.bool_)) or not isinstance(number, Integral) or number < 1
            for number in numbers):
        raise ValueError("EXCLUDED_DATASETS must be a unique list/tuple of positive dataset numbers")
    if len(set(numbers)) != len(numbers):
        raise ValueError("EXCLUDED_DATASETS must not repeat dataset numbers")
    return [int(number) for number in numbers]


def select_catalogue(catalogue, dataset_range, *, excluded_numbers=()):
    """Inclusive stable-number x-y selection, omitting explicitly excluded IDs."""
    rows = _catalogue(catalogue)
    if not isinstance(dataset_range, str) or not re.fullmatch(r"\s*[1-9][0-9]*\s*-\s*[1-9][0-9]*\s*", dataset_range):
        raise ValueError(f"DATASET_RANGE must be an inclusive 'x-y' string between 1 and {len(rows)}")
    start, end = map(int, dataset_range.split("-"))
    if not 1 <= start <= end <= len(rows):
        raise ValueError(f"DATASET_RANGE must satisfy 1 <= x <= y <= {len(rows)}")
    excluded = _excluded_dataset_numbers(excluded_numbers)
    if any(number > len(rows) for number in excluded):
        raise ValueError(f"EXCLUDED_DATASETS contains numbers outside 1-{len(rows)}")
    selected = [row for row in rows[start - 1:end] if row["number"] not in excluded]
    if not selected:
        raise ValueError("No datasets remain in DATASET_RANGE after EXCLUDED_DATASETS; change the range or exclusions")
    return selected


def _configuration(config):
    unknown = set(config) - set(DEFAULTS)
    if unknown:
        raise ValueError(f"Unknown configuration keys: {sorted(unknown)}")
    result = {**DEFAULTS, **config}
    result["EXCLUDED_DATASETS"] = _excluded_dataset_numbers(result["EXCLUDED_DATASETS"])
    if result["SAMPLER_PARAMETERS"] is None:
        raise ValueError("SAMPLER_PARAMETERS must provide exactly smote, kmeans_smote and rwo")
    result["SAMPLER_PARAMETERS"] = validate_sampler_parameters(result["SAMPLER_PARAMETERS"])
    for key in ("N_CANDIDATES", "MAX_ROUNDS", "MLP_EPOCHS", "FLOW_EPOCHS", "N_JOBS"):
        if isinstance(result[key], (bool, np.bool_)) or not isinstance(result[key], Integral) or result[key] < 1:
            raise ValueError(f"{key} must be a positive integer")
    cap = result["AUGMENTATION_CAP"]
    if isinstance(cap, (bool, np.bool_)) or not isinstance(cap, Integral) or cap < 0:
        raise ValueError("AUGMENTATION_CAP must be a nonnegative integer")
    limit = result["MAX_ROWS"]
    if limit is not None and (isinstance(limit, (bool, np.bool_)) or not isinstance(limit, Integral) or limit < 2):
        raise ValueError("MAX_ROWS must be None or an explicit integer of at least two")
    for key in ("TEST_FRACTION", "VALIDATION_FRACTION"):
        if isinstance(result[key], (bool, np.bool_)) or not np.isfinite(result[key]) or not 0 < result[key] < 1:
            raise ValueError(f"{key} must be strictly between zero and one")
    learners = result["CLASSIFIERS"]
    if not isinstance(learners, (list, tuple)) or not learners or len(set(learners)) != len(learners) or set(learners) - set(CLASSIFIERS):
        raise ValueError(f"CLASSIFIERS must be a nonempty unique subset of {CLASSIFIERS}")
    seeds = result["SEEDS"]
    if not isinstance(seeds, (list, tuple)) or not seeds or len(set(seeds)) != len(seeds):
        raise ValueError("SEEDS must be a nonempty unique sequence")
    if any(isinstance(seed, (bool, np.bool_)) or not isinstance(seed, Integral) or not 0 <= seed < 2**32 - 7919 for seed in seeds):
        raise ValueError("SEEDS must be nonnegative integers below 2**32 - 7919")
    for key, lower, upper in (("GROUP_ALPHAS", 0., 1.), ("GATE_TAUS", 0., 1.)):
        values = result[key]
        if not isinstance(values, (list, tuple)) or not values or len(set(values)) != len(values):
            raise ValueError(f"{key} must be a nonempty unique sequence")
        if any(isinstance(v, (bool, np.bool_)) or not np.isfinite(v) or v < lower or v > upper for v in values):
            raise ValueError(f"{key} values must be finite in [{lower}, {upper}]")
        if key == "GATE_TAUS" and any(v in (0, 1) for v in values):
            raise ValueError("GATE_TAUS must be strictly between zero and one")
    if not isinstance(result["SMOKE"], (bool, np.bool_)):
        raise ValueError("SMOKE must be boolean")
    resolve_device(result["DEVICE"])
    for key in ("OUTPUT_DIR", "DATA_HOME"):
        if not isinstance(result[key], (str, Path)) or not str(result[key]):
            raise ValueError(f"{key} must be a nonempty filesystem path")
        result[key] = str(Path(result[key]).expanduser().resolve())
    if result["SMOKE"]:
        result.update(N_CANDIDATES=2, MAX_ROUNDS=2, AUGMENTATION_CAP=40,
                      MLP_EPOCHS=2, FLOW_EPOCHS=2, SEEDS=[int(seeds[0])])
    return result


def _fingerprint(features, target, row_ids):
    digest = hashlib.sha256()
    digest.update(json.dumps([(str(c), str(features[c].dtype)) for c in features]).encode())
    for values in (features, pd.Series(target), pd.Series(row_ids)):
        digest.update(pd.util.hash_pandas_object(values, index=True).to_numpy().tobytes())
    return digest.hexdigest()


def _load_dataset(entry, config):
    """Cache OpenML raw artifacts only; never use CLIMB globally-scaled arrays."""
    import openml
    cache = Path(config["DATA_HOME"]) / "openml"
    cache.mkdir(parents=True, exist_ok=True)
    previous = openml.config.get_config_as_dict()["cachedir"]
    openml.config.set_root_cache_directory(str(cache))
    try:
        raw = openml.datasets.get_dataset(int(entry["openml_id"]), download_data=True,
                                         download_qualities=False)
        features, target, _, _ = raw.get_data(target=raw.default_target_attribute,
            include_row_id=True, dataset_format="dataframe")
        if target is None or np.asarray(target).ndim != 1:
            raise ValueError("Dataset requires exactly one target column")
        target = np.asarray(target)
        if pd.isna(target).any():
            raise ValueError("Raw target contains missing labels")
        y, labels = pd.factorize(target, sort=True)
        y = np.asarray(y, dtype=np.int64)
        if len(labels) != 2:
            raise ValueError(f"Expected a binary CLIMB source, got {len(labels)} classes")
        row_attribute = raw.row_id_attribute
        row_columns = [row_attribute] if isinstance(row_attribute, str) else list(row_attribute or [])
        if row_columns:
            identity_frame, _, _, _ = raw.get_data(include_row_id=True,
                include_ignore_attribute=True, dataset_format="dataframe")
            if len(row_columns) == 1:
                row_ids = identity_frame[row_columns[0]].to_numpy(copy=True)
            else:
                row_ids = np.asarray([json.dumps(list(values), default=_json_default, ensure_ascii=False)
                    for values in identity_frame[row_columns].itertuples(index=False, name=None)])
            features = features.drop(columns=row_columns, errors="ignore")
        else:
            row_ids = np.arange(len(features), dtype=np.int64)
        if pd.isna(row_ids).any() or len(set(row_ids.tolist())) != len(row_ids):
            raise ValueError("Original raw row identities must be nonmissing and unique")
        if features.empty or not len(features.columns):
            raise ValueError("Raw source has no usable features")
        full_fingerprint = _fingerprint(features, target, row_ids)
        metadata = {"source": "openml_raw", "openml_id": int(entry["openml_id"]),
            "catalogue_number": int(entry["number"]), "catalogue_name": entry["name"],
            "imbalance_type": entry["imbalance_type"], "openml_name": raw.name,
            "version": int(raw.version), "openml_md5_checksum": raw.md5_checksum,
            "target_attribute": raw.default_target_attribute, "row_id_attributes": row_columns,
            "row_id_source": "attribute" if row_columns else "original_row_position",
            "raw_fingerprint": full_fingerprint,
            "fingerprint_algorithm": "sha256_pandas_raw_values_v1",
            "label_mapping": [{"label": label, "encoded": index} for index, label in enumerate(labels)],
            "feature_dtypes": {str(column): str(dtype) for column, dtype in features.dtypes.items()},
            "source_n_rows": len(features), "n_features": features.shape[1],
            "explicit_development_row_cap": config["MAX_ROWS"], "development_subset": False}
        limit = config["MAX_ROWS"]
        if limit is not None and limit < len(y):
            chosen, _ = train_test_split(np.arange(len(y)), train_size=int(limit), stratify=y, random_state=42)
            chosen = np.sort(chosen)
            features, target, y, row_ids = features.iloc[chosen].copy(), target[chosen], y[chosen], row_ids[chosen]
            metadata["development_subset"] = True
        metadata.update(n_rows=len(y), data_fingerprint=_fingerprint(features, target, row_ids),
                        class_counts=_counts(y))
        return {**entry, "X": features, "y": y, "row_ids": row_ids, "metadata": metadata}
    finally:
        openml.config.set_root_cache_directory(previous)


def _counts(y):
    classes, counts = np.unique(as_array(y), return_counts=True)
    return {int(label): int(count) for label, count in zip(classes, counts)}


def _split(dataset, seed, config):
    rows = np.arange(len(dataset["y"]))
    train, untouched_test = train_test_split(rows, test_size=config["TEST_FRACTION"],
        stratify=dataset["y"], random_state=seed)
    if config["SMOKE"]:
        # The study's outer test remains entirely unconsumed in smoke.
        if len(train) > 1200:
            subset, _ = train_test_split(train, train_size=1200,
                stratify=dataset["y"][train], random_state=seed)
        else:
            subset = train
        train, test = train_test_split(subset, test_size=config["TEST_FRACTION"],
            stratify=dataset["y"][subset], random_state=seed)
    else:
        test = untouched_test
    fit, validation = train_test_split(train, test_size=config["VALIDATION_FRACTION"],
        stratify=dataset["y"][train], random_state=seed)
    result = {"train": train, "fit": fit, "validation": validation, "test": test}
    if any(len(np.unique(dataset["y"][indices])) != 2 for indices in result.values()):
        raise ValueError("Every real split must contain both classes")
    if config["SMOKE"] and set(test.tolist()) & set(untouched_test.tolist()):
        raise ValueError("Smoke must not consume any scientific outer-test row")
    return result


def _partitions(dataset, split):
    inner = make_preprocessor(dataset["X"].iloc[split["fit"]])
    outer = make_preprocessor(dataset["X"].iloc[split["train"]])
    return {"inner": inner, "outer": outer,
        "X_fit": inner.fit_transform(dataset["X"].iloc[split["fit"]]).astype(np.float32),
        "X_validation": inner.transform(dataset["X"].iloc[split["validation"]]).astype(np.float32),
        "X_train": outer.fit_transform(dataset["X"].iloc[split["train"]]).astype(np.float32),
        "X_test": outer.transform(dataset["X"].iloc[split["test"]]).astype(np.float32)}


def _categorical_groups(preprocessor):
    """Recover each complete fitted one-hot variable, not one merged block."""
    columns = preprocessor.named_steps["columns"]
    block = columns.output_indices_["categorical"]
    if block.start == block.stop:
        return ()
    encoder = columns.named_transformers_["categorical"].named_steps["encode"]
    groups, start = [], block.start
    for categories in encoder.categories_:
        stop = start + len(categories)
        groups.append(tuple(range(start, stop)))
        start = stop
    if start != block.stop:
        raise ValueError("One-hot category widths do not match the fitted output block")
    return tuple(groups)


def _targets(y, cap):
    """Matched total deficit budget, clipped at real majority balance."""
    counts = _counts(y)
    maximum = max(counts.values())
    deficits = {label: maximum - count for label, count in counts.items()}
    total = min(int(cap), sum(deficits.values()))
    if not total:
        return counts
    desired = {label: total * value / sum(deficits.values()) for label, value in deficits.items()}
    allocated = {label: int(np.floor(value)) for label, value in desired.items()}
    for label in sorted(counts, key=lambda c: (-(desired[c] - allocated[c]), c))[:total - sum(allocated.values())]:
        allocated[label] += 1
    return {label: counts[label] + allocated[label] for label in counts}


def choose_f1_threshold(y_true, probability):
    """O(n log n), exact rational F1 comparisons over whole tied-score groups.

    Candidates are observed score cut points and .5. Exact F1 ties prefer the
    threshold closest to .5, then the higher threshold (never split a score tie).
    """
    y = np.asarray(y_true, dtype=np.int64)
    p = np.asarray(probability, dtype=np.float64)
    if y.ndim != 1 or p.shape != y.shape or not len(y):
        raise ValueError("Threshold selection requires aligned nonempty vectors")
    if not np.isin(y, [0, 1]).all() or not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
        raise ValueError("Threshold selection requires binary labels and finite probabilities")
    positives = int(y.sum())
    if not positives:
        raise ValueError("Validation must contain the minority class")
    order = np.argsort(-p, kind="stable")
    ordered = p[order]
    ends = np.r_[np.flatnonzero(ordered[:-1] != ordered[1:]), len(y) - 1]
    cumulative = np.cumsum(y[order])
    at_half = p >= .5
    best = (int(y[at_half].sum()), int(at_half.sum()) + positives, .5)
    for end in ends:
        item = (int(cumulative[end]), int(end) + 1 + positives, float(ordered[end]))
        comparison = item[0] * best[1] - best[0] * item[1]
        if comparison > 0 or (comparison == 0 and
                (-abs(item[2] - .5), item[2]) > (-abs(best[2] - .5), best[2])):
            best = item
    return best[2], 2 * best[0] / best[1]


def _metrics(y, probability, threshold):
    truth = np.asarray(y, dtype=np.int64)
    predicted = np.asarray(probability) >= threshold
    tn, fp, fn, tp = confusion_matrix(truth, predicted, labels=[0, 1]).ravel()
    return {"minority_f1": float(f1_score(truth, predicted, zero_division=0)),
        "minority_precision": float(precision_score(truth, predicted, zero_division=0)),
        "minority_recall": float(recall_score(truth, predicted, zero_division=0)),
        "average_precision": float(average_precision_score(truth, probability)),
        "balanced_accuracy": float(balanced_accuracy_score(truth, predicted)),
        "mcc": float(matthews_corrcoef(truth, predicted)),
        "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn),
        "f1_at_0_5": float(f1_score(truth, np.asarray(probability) >= .5, zero_division=0))}


def _positive_probability(model, X, positive):
    columns = np.flatnonzero(np.asarray(model.classes_) == positive)
    if len(columns) != 1:
        raise ValueError("Learner has no unique minority-class probability column")
    probabilities = np.asarray(model.predict_proba(X), dtype=np.float64)
    if probabilities.ndim != 2 or len(probabilities) != len(X):
        raise ValueError("Learner produced misaligned class probabilities")
    probability = probabilities[:, columns[0]]
    if not np.isfinite(probability).all() or ((probability < 0) | (probability > 1)).any():
        raise ValueError("Learner produced invalid minority probabilities")
    return probability


def _compact(value):
    """No row arrays, proposal logs, checkpoints or large group vectors on disk."""
    if isinstance(value, dict):
        result = {}
        for key, item in value.items():
            if key in ("candidate_log", "provenance", "training_losses", "fit_row_ids",
                       "calibration_row_ids", "validation_row_ids", "real_row_ids"):
                continue
            if key in ("seed_multiplicity", "attempted_seed_multiplicity"):
                multiplicities = [int(count) for count in item.values()]
                result[str(key)] = {"unique_seed_count": len(multiplicities),
                    "total": sum(multiplicities), "maximum": max(multiplicities, default=0),
                    "sha256": _hash(item)}
            else:
                result[str(key)] = _compact(item)
        return result
    if isinstance(value, np.ndarray):
        return {"count": len(value), "sha256": _hash(value)}
    if isinstance(value, (list, tuple)):
        if len(value) > 32:
            return {"count": len(value), "sha256": _hash(value)}
        return [_compact(item) for item in value]
    return value


def _audit(sampled, y, row_ids, targets, real_only, gate=None, tau=None):
    labels = as_array(sampled.y)
    n = len(y)
    provenance = sampled.provenance
    accepted = len(labels) - n
    requested = sum(targets.values()) - n
    if len(provenance) != accepted or not 0 <= accepted <= requested:
        raise ValueError("Augmentation counts do not match ancestry/cap")
    if not np.array_equal(labels[:n], y):
        raise ValueError("Augmentation changed real labels/order")
    ancestry, used_roots = [], set()
    direct, synthetic = 0, 0
    for offset, entry in enumerate(provenance):
        row = n + offset
        if int(entry["row_index"]) != row or int(entry["class"]) != int(labels[row]):
            raise ValueError("Synthetic ancestry is not contiguous and class-aligned")
        parents = [int(entry["seed_index"]), int(entry["partner_index"])]
        relocation = int(entry.get("relocation_index", -1))
        if relocation >= 0:
            parents.append(relocation)
        roots = set()
        for parent in parents:
            if not 0 <= parent < row or (real_only and parent >= n):
                raise ValueError("Synthetic parent outside allowed training ancestry")
            if labels[parent] != labels[row]:
                raise ValueError("Synthetic parent has the wrong class")
            direct += parent < n
            synthetic += parent >= n
            roots.update({parent} if parent < n else ancestry[parent - n])
        ancestry.append(roots)
        used_roots.update(roots)
    if any(_counts(labels).get(label, 0) > maximum for label, maximum in targets.items()):
        raise ValueError("Sampler exceeded a per-class target")
    log = sampled.support_summary.get("candidate_log")
    gate_evidence = None
    if log is not None:
        if len(log) > requested:
            raise ValueError("Proposals exceeded the matched augmentation budget")
        for proposal in log:
            for key in ("seed_index", "partner_index"):
                parent = int(proposal[key])
                if not 0 <= parent < n or int(labels[parent]) != int(proposal["class"]):
                    raise ValueError("Proposal parent is not same-class real training")
            if gate is not None and proposal["accepted"]:
                ratio = proposal.get("log_ratio")
                if ratio is None or not np.isfinite(ratio) or ratio < np.log(tau) - np.log1p(-tau):
                    raise ValueError("Accepted proposal log violates the fixed log-ratio threshold")
        accepted_entries = [item for item in log if item["accepted"]]
        if len(accepted_entries) != accepted:
            raise ValueError("Candidate acceptance log does not reconcile final ancestry")
        for proposal, final in zip(accepted_entries, provenance):
            for key in ("seed_index", "partner_index", "class", "proposal_id"):
                if proposal[key] != final[key]:
                    raise ValueError("Accepted proposal identity differs from final ancestry")
        budget = sampled.support_summary["proposal_budget"]
        if (budget["attempted"] != len(log) or budget["accepted"] != accepted or
                budget["rejected"] != len(log) - accepted):
            raise ValueError("Attempt/accept/reject quota accounting does not reconcile")
        groups = sampled.support_summary["group_support"]
        if any(int(count) > 4 for count in groups["attempted_seed_multiplicity"].values()):
            raise ValueError("U2 attempted seed multiplicity exceeds canonical cap 4")
        for group in groups["groups"]:
            if int(group["attempted_generated"]) > 2 * len(group["indices"]):
                raise ValueError("U2 attempted group quota exceeds canonical cap 2n")
    if gate is not None:
        for entry in provenance:
            if int(entry.get("relocation_index", -1)) != -1:
                raise ValueError("FlowGate must replace native relocation")
        if accepted:
            checked = gate.predict(as_array(sampled.X[n:]), labels[n:], tau)
            if not bool(np.asarray(checked["accepted"]).all()):
                raise ValueError("Final synthetic coordinates violate the frozen FlowGate")
        gate_evidence = {"accepted_coordinates_checked": accepted, "all_pass": True,
                         "tau": tau, "no_relocation": True}
    root_ids = np.asarray(row_ids)[sorted(used_roots)]
    return {"accepted_count": accepted, "requested_count": requested,
        "underspend": requested - accepted, "real_count": n, "real_counts": _counts(y),
        "actual_counts": _counts(labels), "stop_reason": sampled.stop_reason,
        "round_count": len(sampled.rounds), "synthetic_weights": 1,
        "support_summary": _compact(sampled.support_summary),
        "ancestry": {"validated": True, "real_only_parents": real_only,
            "direct_real_parent_references": int(direct), "synthetic_parent_references": int(synthetic),
            "unique_real_root_count": len(used_roots), "real_root_row_ids_sha256": _hash(root_ids),
            "training_row_ids_sha256": _hash(np.asarray(row_ids)),
            "provenance_sha256": _hash(provenance)}, "gate_acceptance": gate_evidence}


def _fit(dataset, split, partitions, classifier, method, parameters, sampler, config,
         seed, role, prepared, gate=None):
    indices = split["fit" if role == "inner" else "train"]
    features = partitions["X_fit" if role == "inner" else "X_train"]
    y = dataset["y"][indices]
    targets = _targets(y, config["AUGMENTATION_CAP"])
    fits = 0
    latest_model, latest_count = None, None
    def train_round(X, labels):
        nonlocal fits, latest_model, latest_count
        model = make_estimator(classifier, seed, parameters, device=config["DEVICE"],
            epochs=config["MLP_EPOCHS"], n_jobs=config["N_JOBS"])
        model = fit_estimator(model, X, labels)
        fits += 1
        latest_model, latest_count = model, len(labels)
        return model
    if method == "baseline":
        model = train_round(features, y)
        return model, {"accepted_count": 0, "requested_count": 0, "real_count": len(y),
            "real_counts": _counts(y), "actual_counts": _counts(y), "classifier_fit_count": fits,
            "stop_reason": "no_oversampling", "synthetic_weights": 1}
    if method in SAMPLER_PARAMETERS:
        # Fixed training-only pools are identical across learner candidates/families.
        pools = prepared.setdefault("control_pools", {})
        key = (role, method)
        if key not in pools:
            started = perf_counter()
            X_res, y_res, audit = resample_control(method, features, y, targets, seed,
                categorical_groups=_categorical_groups(partitions[role]),
                n_jobs=config["N_JOBS"], parameters=sampler)
            audit = {**audit, "pool_scope": "dataset_seed_role_method",
                "pool_role": role, "pool_seed": seed,
                "shared_generation_seconds": perf_counter() - started}
            pools[key] = X_res, y_res, audit
        X_res, y_res, audit = pools[key]
        model = train_round(X_res, y_res)
        return model, {**audit, "classifier_fit_count": fits}
    categorical = categorical_columns(partitions[role])
    if method == "original_ugo":
        options = OriginalUGOConfig(max_iterations=config["MAX_ROUNDS"], random_state=seed,
            device=config["DEVICE"], categorical_columns=categorical,
            categorical_projection=bool(categorical))
        sampled = original_ugo(features, y, train_round, options, target_counts=targets)
    elif method in ("u2_native", "u2_flow_gate"):
        if role not in prepared:
            prepared[role] = prepare_u2(features, y, seed, targets, sampler)
        sampled = run_u2(features, y, train_round, seed=seed, targets=targets,
            parameters=sampler, max_iterations=config["MAX_ROUNDS"], device=config["DEVICE"],
            categorical_columns=categorical, prepared=prepared[role], flow_support=gate)
    else:
        raise ValueError(f"Unknown benchmark method: {method}")
    unchanged = method == "u2_flow_gate" and latest_count == len(sampled.y)
    model = latest_model if unchanged else train_round(sampled.X, sampled.y)
    audit = _audit(sampled, y, dataset["row_ids"][indices], targets,
                   method != "original_ugo", gate, sampler.get("f1_tau"))
    audit["classifier_fit_count"] = fits
    return model, audit


def _candidate(dataset, split, partitions, classifier, method, parameters, sampler,
               config, seed, positive, candidate_id, prepared, gate=None):
    try:
        model, audit = _fit(dataset, split, partitions, classifier, method, parameters, sampler,
                           config, seed, "inner", prepared, gate)
        probability = _positive_probability(model, partitions["X_validation"], positive)
        truth = dataset["y"][split["validation"]] == positive
        threshold, score = choose_f1_threshold(truth, probability)
        metrics = _metrics(truth, probability, threshold)
        return {"candidate_id": candidate_id, "parameters": parameters,
            "sampler_parameters": sampler, "threshold": threshold, "validation_f1": score,
            "validation_ap": metrics["average_precision"],
            "f1_numerator": 2 * metrics["tp"],
            "f1_denominator": 2 * metrics["tp"] + metrics["fp"] + metrics["fn"],
            "audit": audit}, probability
    except Exception as error:
        context = {"role": "inner", "method": method, "candidate_id": candidate_id,
                   "parameters": parameters, "sampler_parameters": sampler}
        raise RuntimeError(f"Native candidate failed: {json.dumps(context, default=_json_default)}") from error


def _better(item, best):
    if best is None:
        return True
    comparison = item["f1_numerator"] * best["f1_denominator"] - best["f1_numerator"] * item["f1_denominator"]
    return comparison > 0 or (comparison == 0 and item["validation_ap"] > best["validation_ap"])


def _core_hash(core):
    digest = hashlib.sha256()
    for name, value in core.model.state_dict().items():
        digest.update(name.encode())
        digest.update(value.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()


def _flow_roles(dataset, split, partitions, seed, config):
    fit, calibration = train_test_split(split["fit"], test_size=.2,
        stratify=dataset["y"][split["fit"]], random_state=seed + 7919)
    core = fit_flow_gmm(dataset["X"].iloc[fit], dataset["y"][fit],
        dataset["row_ids"][fit], seed, FlowGMMConfig(epochs=config["FLOW_EPOCHS"], device=config["DEVICE"]))
    gates = []
    for role in ("inner", "outer"):
        gate = FlowGate(core, partitions[role], calibration_row_ids=dataset["row_ids"][calibration],
                        validation_row_ids=dataset["row_ids"][split["validation"]])
        gate.role = role
        gate.calibration_class_counts = _counts(dataset["y"][calibration])
        gates.append(gate)
    evidence = _flow_evidence(gates, dataset, split)
    return gates, evidence


def _flow_evidence(gates, dataset, split):
    inner, outer = gates
    core = inner.core
    if outer.core is not core or core.model.training or any(p.requires_grad for p in core.model.parameters()):
        raise ValueError("Flow core must remain frozen and shared across outer refit")
    roles = [set(np.asarray(ids).tolist()) for ids in (core.fit_row_ids,
        inner.calibration_row_ids, inner.validation_row_ids, dataset["row_ids"][split["test"]])]
    if any(roles[a] & roles[b] for a in range(4) for b in range(a + 1, 4)):
        raise ValueError("Flow fit/calibration/validation/test roles overlap")
    if roles[0] | roles[1] != set(dataset["row_ids"][split["fit"]].tolist()):
        raise ValueError("Flow fit and reserved calibration do not partition inner real fit")
    if roles[2] != set(dataset["row_ids"][split["validation"]].tolist()):
        raise ValueError("Flow validation identities do not match real validation")
    if not np.array_equal(outer.calibration_row_ids, inner.calibration_row_ids):
        raise ValueError("Outer replay changed reserved flow calibration identities")
    audit = core.audit
    if audit["synthetic_training_count"] != 0 or audit["calibration_used"] or not audit["frozen"]:
        raise ValueError("Flow must be fitted only on real fit rows and frozen")
    current_hash = _core_hash(core)
    if current_hash != audit["model_checksum"]:
        raise ValueError("Flow state changed since the real-only fit")
    return {"core_sha256": current_hash, "fit_row_ids_sha256": _hash(core.fit_row_ids),
        "calibration_row_ids_sha256": _hash(inner.calibration_row_ids),
        "validation_row_ids_sha256": _hash(inner.validation_row_ids),
        "fit_count": len(core.fit_row_ids), "calibration_count": len(inner.calibration_row_ids),
        "validation_count": len(inner.validation_row_ids), "roles_disjoint": True,
        "fit_count_shared": 1, "frozen_core_shared_across_roles": True,
        "fit_seconds": float(audit["training_seconds"]), "calibration_used": False,
        "synthetic_training_count": 0, "resolved_device": audit["resolved_device"],
        "model_config": audit["model_config"], "source_sha256": audit["source_sha256"]}


def _safe_name(entry, classifier, seed, method):
    name = re.sub(r"[^A-Za-z0-9_.-]+", "_", entry["name"]).strip("._-")[:70] or "dataset"
    return f"{int(entry['number']):02d}_{name}__{classifier}__s{seed}__{method}"


def _predictions(path, row_ids, labels, probability, threshold, positive):
    pd.DataFrame({"row_id": row_ids, "y_true": (labels == positive).astype(np.int64),
        "y_true_encoded": labels, "minority_class": positive, "probability": probability,
        "y_pred": (probability >= threshold).astype(np.int64),
        "y_pred_encoded": np.where(probability >= threshold, positive, 1 - positive),
        "threshold": threshold}).to_csv(path, index=False, float_format="%.17g")


def _cell(dataset, split, partitions, classifier, seed, method, config, candidates,
          run_dir, prepared, gates=None, flow_evidence=None):
    started = perf_counter()
    counts = _counts(dataset["y"][split["train"]])
    positive = min(counts, key=lambda label: (counts[label], label))
    selection = {"dataset": dataset["name"], "dataset_number": dataset["number"],
        "classifier": classifier, "seed": seed, "method": method,
        "positive_class": positive, "canonical_candidates": [], "candidates": [],
        "selection_objective": "validation_minority_f1_then_average_precision",
        "threshold_policy": "whole_tied_score_groups_plus_0.5_exact_rational_f1_ties_nearest_0.5_then_higher"}
    best, best_probability = None, None
    if method not in METHODS:
        raise ValueError(f"Unknown benchmark method: {method}")
    if method != "u2_flow_gate":
        sampler = config["SAMPLER_PARAMETERS"][method] if method in SAMPLER_PARAMETERS else {}
        for index, parameters in enumerate(candidates):
            item, probability = _candidate(dataset, split, partitions, classifier, method,
                parameters, sampler, config, seed, positive, index, prepared)
            selection["candidates"].append(item)
            if _better(item, best):
                best, best_probability = item, probability
        selection["selected_candidate_id"] = best["candidate_id"]
    else:
        locked = None
        for index, parameters in enumerate(candidates):
            sampler = {"group_alpha": config["GROUP_ALPHAS"][index % len(config["GROUP_ALPHAS"]) ]}
            item, _ = _candidate(dataset, split, partitions, classifier, "u2_native", parameters,
                sampler, config, seed, positive, index, prepared)
            selection["canonical_candidates"].append(item)
            if _better(item, locked):
                locked = item
        selection["selected_candidate_id"] = locked["candidate_id"]
        selection["locked_learner_parameters"] = locked["parameters"]
        selection["locked_group_alpha"] = locked["sampler_parameters"]["group_alpha"]
        for index, tau in enumerate(config["GATE_TAUS"]):
            sampler = {**locked["sampler_parameters"], "f1_tau": tau}
            item, probability = _candidate(dataset, split, partitions, classifier, method,
                locked["parameters"], sampler, config, seed, positive, index, prepared, gates[0])
            selection["candidates"].append(item)
            if _better(item, best):
                best, best_probability = item, probability
        selection["selected_tau_id"] = best["candidate_id"]
        selection["flow_evidence"] = flow_evidence
    threshold = best["threshold"]
    basename = _safe_name(dataset, classifier, seed, method)
    validation_path = Path("predictions") / f"{basename}__validation.csv"
    test_path = Path("predictions") / f"{basename}__test.csv"
    # Preserve actual selected inner predictions before any outer model exists.
    _predictions(run_dir / validation_path, dataset["row_ids"][split["validation"]],
        dataset["y"][split["validation"]], best_probability, threshold, positive)
    try:
        model, outer_audit = _fit(dataset, split, partitions, classifier, method, best["parameters"],
            best["sampler_parameters"], config, seed, "outer", prepared,
            gates[1] if method == "u2_flow_gate" else None)
        probability = _positive_probability(model, partitions["X_test"], positive)
    except Exception as error:
        context = {"role": "outer", "parameters": best["parameters"],
                   "sampler_parameters": best["sampler_parameters"], "frozen_threshold": threshold}
        raise RuntimeError(f"Selected outer refit failed: {json.dumps(context, default=_json_default)}") from error
    if method == "u2_flow_gate" and _flow_evidence(gates, dataset, split) != flow_evidence:
        raise ValueError("Frozen detector evidence changed during inner selection/outer refit")
    _predictions(run_dir / test_path, dataset["row_ids"][split["test"]],
        dataset["y"][split["test"]], probability, threshold, positive)
    fits = sum(item["audit"]["classifier_fit_count"] for item in
        selection["canonical_candidates"] + selection["candidates"]) + outer_audit["classifier_fit_count"]
    selection.update(selected_parameters=best["parameters"], selected_sampler_parameters=best["sampler_parameters"],
        threshold=threshold, selected_validation_f1=best["validation_f1"],
        selected_validation_ap=best["validation_ap"], selected_inner_audit=best["audit"],
        outer_audit=outer_audit, classifier_fit_count=fits, backend=model.backend_,
        prediction_paths={"validation": str(validation_path), "test": str(test_path)})
    split_hash = _hash({role: dataset["row_ids"][indices] for role, indices in split.items()})
    record = {"dataset_number": dataset["number"], "dataset": dataset["name"],
        "openml_id": dataset["openml_id"], "classifier": classifier, "seed": seed,
        "method": method, "status": "completed", "run_kind": "training_only_smoke" if config["SMOKE"] else "study",
        "development_subset": dataset["metadata"]["development_subset"], "positive_class": positive,
        "threshold": threshold, **_metrics(dataset["y"][split["test"]] == positive, probability, threshold),
        "validation_f1": best["validation_f1"], "validation_ap": best["validation_ap"],
        "accepted_count": outer_audit["accepted_count"], "requested_count": outer_audit["requested_count"],
        "n_inner_fit": len(split["fit"]), "n_validation": len(split["validation"]),
        "n_outer_train": len(split["train"]), "n_test": len(split["test"]),
        "runtime_seconds": perf_counter() - started, "classifier_fit_count": fits,
        "backend": model.backend_,
        "sampler_device": "cpu" if method in SAMPLER_PARAMETERS else resolve_device(config["DEVICE"]),
        "flow_fit_seconds_shared": flow_evidence["fit_seconds"] if method == "u2_flow_gate" else 0.,
        "split_fingerprint": split_hash, "data_fingerprint": dataset["metadata"]["data_fingerprint"],
        "predictions_path": str(test_path), "validation_predictions_path": str(validation_path)}
    return record, selection


def _tables(records, selected, config):
    """Keep missing coverage explicit; deltas always mean method minus reference."""
    results = pd.DataFrame(records)
    rows, comparison, paired = [], [], []
    lookup = {(row["dataset_number"], row["classifier"], row["seed"], row["method"]): row for row in records}
    run_kind = "training_only_smoke" if config["SMOKE"] else "study"
    expected = len(config["SEEDS"])
    for entry in selected:
        for classifier in config["CLASSIFIERS"]:
            identity = {"dataset_number": entry["number"], "dataset": entry["name"],
                        "classifier": classifier, "run_kind": run_kind}
            wide = dict(identity)
            complete = True
            for method in METHODS:
                subset = [lookup[(entry["number"], classifier, seed, method)]
                    for seed in config["SEEDS"]
                    if (entry["number"], classifier, seed, method) in lookup]
                row = {**identity, "method": method, "expected_seeds": expected,
                    "completed_seeds": len(subset), "failed_or_missing_seeds": expected - len(subset),
                    "coverage_complete": len(subset) == expected,
                    "status": "complete" if len(subset) == expected else "incomplete"}
                for metric in METRICS:
                    values = np.asarray([item[metric] for item in subset], dtype=float)
                    row[metric + "_mean"] = float(values.mean()) if len(values) else np.nan
                    row[metric + "_std"] = float(values.std(ddof=1)) if len(values) > 1 else np.nan
                rows.append(row)
                for field in ("minority_f1_mean", "minority_f1_std", "completed_seeds",
                              "expected_seeds", "failed_or_missing_seeds", "coverage_complete"):
                    wide[f"{method}_{field}"] = row[field]
                complete = complete and row["coverage_complete"]
            # No winner/rank is inferred from incomplete or differently covered means.
            wide.update(coverage_complete=complete, status="complete" if complete else "incomplete")
            comparison.append(wide)
            for seed in config["SEEDS"]:
                for index, reference in enumerate(METHODS):
                    for method in METHODS[index + 1:]:
                        row = lookup.get((entry["number"], classifier, seed, method))
                        other = lookup.get((entry["number"], classifier, seed, reference))
                        matched = row is not None and other is not None
                        if matched and (row["split_fingerprint"] != other["split_fingerprint"] or
                                        row["data_fingerprint"] != other["data_fingerprint"]):
                            raise ValueError("Paired comparison attempted across different real data or splits")
                        paired.append({**identity, "seed": seed, "method": method,
                            "reference": reference, "status": "complete" if matched else "incomplete",
                            "method_completed": row is not None, "reference_completed": other is not None,
                            **{metric + "_delta": row[metric] - other[metric] if matched else np.nan
                               for metric in METRICS},
                            "split_fingerprint": row["split_fingerprint"] if matched else None,
                            "data_fingerprint": row["data_fingerprint"] if matched else None})
    paired_columns = ["dataset_number", "dataset", "classifier", "seed", "method", "reference",
        "run_kind", "status", "method_completed", "reference_completed"] + [
        metric + "_delta" for metric in METRICS] + ["split_fingerprint", "data_fingerprint"]
    if results.empty:
        results = pd.DataFrame(columns=["dataset_number", "dataset", "classifier", "seed", "method", "status"] + list(METRICS))
    return results, pd.DataFrame(rows), pd.DataFrame(paired, columns=paired_columns), pd.DataFrame(comparison)


def _dependencies():
    versions = {}
    for name in ("numpy", "pandas", "scipy", "scikit-learn", "imbalanced-learn", "torch", "xgboost", "openml", "threadpoolctl"):
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = "not_installed"
    return versions


def _source_hashes():
    directory = Path(__file__).resolve().parent
    return {path.name: hashlib.sha256(path.read_bytes()).hexdigest() for path in sorted(directory.glob("*.py"))}


def run_benchmark(config, catalogue):
    """Run every requested real-data cell, flushing success/failure evidence per cell.

    Returns run_dir and results/summary/paired_deltas/comparison DataFrames.
    Dataset or training failures are explicit failures.json entries; they never
    change learners or sampler settings, omit gating, fabricate metrics or stop other cells.
    """
    config = _configuration(dict(config))
    selected = select_catalogue(catalogue, config["DATASET_RANGE"],
                                excluded_numbers=config["EXCLUDED_DATASETS"])
    root = Path(config["OUTPUT_DIR"])
    root.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    run_dir = root / f"{timestamp}_{'smoke' if config['SMOKE'] else 'study'}_{uuid.uuid4().hex[:10]}"
    run_dir.mkdir()
    (run_dir / "predictions").mkdir()
    started = perf_counter()
    expected = len(selected) * len(config["CLASSIFIERS"]) * len(config["SEEDS"]) * len(METHODS)
    manifest = {"created_utc": datetime.now(timezone.utc).isoformat(), "config": config,
        "selected_catalogue": selected, "methods": list(METHODS), "dependencies": _dependencies(),
        "source_sha256": _source_hashes(), "python": sys.version, "platform": platform.platform(),
        "cuda": {"available": torch.cuda.is_available(), "torch_cuda_version": torch.version.cuda,
                 "resolved_device": resolve_device(config["DEVICE"]),
                 "device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None},
        "backend_policy": {"mlp": "pytorch_DEVICE", "xgboost": "xgboost_DEVICE_hist",
            "logistic_regression": "sklearn_cpu", "random_forest": "sklearn_cpu",
            "svm": "sklearn_cpu_probability_training_calibration",
            "control_samplers": "smote_kmeans_smote_rwo_cpu; downstream_native_learner_DEVICE_unchanged"},
        "scientific_result": not config["SMOKE"] and config["MAX_ROWS"] is None,
        "run_kind": "training_only_smoke" if config["SMOKE"] else "study",
        "datasets": [], "coverage": {"expected_cells": expected, "completed_cells": 0, "complete": False},
        "status": "running", "runtime_seconds": 0., "shared_flow_fits": [],
        "protocol": {"validation": "real_inner_holdout_only", "threshold": "frozen_before_outer_refit",
            "minority_class": "least_frequent_encoded_class_in_real_outer_train",
            "selection": "minority_F1_primary_AP_secondary", "mlp": "fixed_epochs_no_validation_stopping",
            "synthetic_weights": 1, "raw_cache_only": True,
            "original_ugo": "paper_derived_reference_not_author_reproduction; mixed_features_use_explicit_partner_category_projection",
            "flow": "single_real_inner_fit_core_per_dataset_seed_reserved_calibration_unused_frozen_outer_bridge",
            "control_samplers": {
                "parameters": config["SAMPLER_PARAMETERS"],
                "tuning": "same_native_learner_candidates_and_F1_threshold_opportunities_as_baseline",
                "generation": "real_role_training_only_once_per_dataset_seed_role_method; shared_in_memory_across_candidates_and_classifiers",
                "runtime": "cell_wall_time_includes_pool_generation_only_for_first_consumer; audit_shared_generation_seconds_is_not_a_per_cell_fit",
                "weights": 1,
                "targets": "same_real_class_deficit_targets_and_augmentation_cap_as_UGO",
                "mixed_data_adapter": "SMOTE_and_KMeansSMOTE_on_numeric_plus_full_onehot_coordinates_then_argmax_each_whole_nominal_block_for_synthetics_only; not_SMOTENC",
                "kmeans_smote_budget": "deterministically_trim_ceiling_overproduction_only; never_refill",
                "default_cluster_balance_threshold": "prespecified_inclusive_0.0_allows_rare_positive_clusters; not_a_fallback_or_dataset_search",
                "rwo": "independent_real_seed_plus_Normal_times_real_class_population_std_over_sqrt_n; empirical_real_class_nominal_roulette; no_clipping_rounding_or_chaining",
                "rwo_std_reference": "https://raw.githubusercontent.com/analyticalmindsltd/smote_variants/master/smote_variants/oversampling/_rwo_sampling.py",
                "rwo_nominal_reference": "https://raw.githubusercontent.com/cran/imbalance/master/R/rwo.R",
                "rwo_reference_difference": "independently_written; population_std_convention_follows_Python_reference_not_bitwise_R_variance_variant",
                "failure_semantics": "infeasible_neighbors_clusters_or_other_native_errors_are_logged_failed_cells; no_ROS_fallback_parameter_relaxation_or_synthetic_metrics",
                "ancestry": "imblearn_parent_indices_not_exposed_and_not_fabricated"},
            "paired_deltas": "all_unordered_METHODS_pairs_in_method_order; later_method_minus_earlier_reference; identical_data_and_split_required; missing_pairs_have_NaN_deltas",
            "comparison": "per_dataset_classifier_F1_mean_sample_std_and_completed_expected_seed_coverage_for_every_method; no_partial_coverage_winner",
            "smoke": "all_smoke_observations_drawn_only_from_scientific_outer_train"}}
    records, selections, failures = [], {}, []
    def flush():
        results, summary, paired, comparison = _tables(records, selected, config)
        results.to_csv(run_dir / "per_cell_metrics.csv", index=False, float_format="%.17g")
        summary.to_csv(run_dir / "summary.csv", index=False, float_format="%.17g")
        paired.to_csv(run_dir / "paired_deltas.csv", index=False, float_format="%.17g")
        comparison.to_csv(run_dir / "comparison.csv", index=False, float_format="%.17g")
        manifest["coverage"] = {"expected_cells": expected, "completed_cells": len(records),
            "failed_or_missing_cells": expected - len(records), "complete": len(records) == expected,
            "failure_events": len(failures)}
        manifest["runtime_seconds"] = perf_counter() - started
        _dump(run_dir / "manifest.json", manifest)
        _dump(run_dir / "selections.json", selections)
        _dump(run_dir / "failures.json", failures)
        return results, summary, paired, comparison
    def failed(kind, context, error):
        failures.append({"kind": kind, **context, "exception_type": type(error).__name__,
                         "message": str(error), "traceback": traceback.format_exc()})
        print(f"FAILED {kind}: {context}: {type(error).__name__}: {error}", flush=True)
        flush()
    flush()
    original_threads = torch.get_num_threads()
    torch.set_num_threads(config["N_JOBS"])
    try:
        with threadpool_limits(limits=config["N_JOBS"]):
            for entry in selected:
                context = {"dataset_number": entry["number"], "dataset": entry["name"], "openml_id": entry["openml_id"]}
                try:
                    dataset = _load_dataset(entry, config)
                    manifest["datasets"].append(dataset["metadata"])
                    flush()
                except Exception as error:
                    failed("dataset_error", {**context, "affected_cells": len(config["CLASSIFIERS"]) * len(config["SEEDS"]) * len(METHODS)}, error)
                    continue
                for seed in config["SEEDS"]:
                    try:
                        split = _split(dataset, seed, config)
                        partitions = _partitions(dataset, split)
                    except Exception as error:
                        failed("dataset_seed_error", {**context, "seed": seed,
                            "affected_cells": len(config["CLASSIFIERS"]) * len(METHODS)}, error)
                        continue
                    prepared, gates, flow_evidence, flow_error = {}, None, None, None
                    for classifier in config["CLASSIFIERS"]:
                        candidates = candidate_parameters(classifier, seed, config["N_CANDIDATES"], config["SMOKE"])
                        for method in METHODS:
                            cell_context = {**context, "classifier": classifier, "seed": seed, "method": method,
                                "candidate_configs": candidates, "config": config}
                            try:
                                if method == "u2_flow_gate":
                                    if flow_error is not None:
                                        raise RuntimeError(f"Shared real-only FlowGMM fit failed: {flow_error}")
                                    if gates is None:
                                        try:
                                            gates, flow_evidence = _flow_roles(dataset, split, partitions, seed, config)
                                            manifest["shared_flow_fits"].append({
                                                "dataset_number": entry["number"], "dataset": entry["name"],
                                                "seed": seed, "fit_seconds": flow_evidence["fit_seconds"],
                                                "core_sha256": flow_evidence["core_sha256"], "physical_fit_count": 1})
                                        except Exception as error:
                                            flow_error = f"{type(error).__name__}: {error}"
                                            raise
                                record, selection = _cell(dataset, split, partitions, classifier, seed,
                                    method, config, candidates, run_dir, prepared, gates, flow_evidence)
                                records.append(record)
                                selections[_safe_name(dataset, classifier, seed, method)] = selection
                                flush()
                                print(f"{entry['number']:02d} {entry['name']} | {classifier} | seed {seed} | {method}: "
                                    f"F1={record['minority_f1']:.4f} AP={record['average_precision']:.4f} "
                                    f"accepted={record['accepted_count']}/{record['requested_count']} backend={record['backend']}", flush=True)
                            except Exception as error:
                                failed("cell_error", cell_context, error)
                    del partitions, prepared, gates
    finally:
        torch.set_num_threads(original_threads)
    manifest["status"] = "completed" if len(records) == expected else "incomplete"
    results, summary, paired, comparison = flush()
    coverage = manifest["coverage"]
    print(f"Saved {run_dir}\nCoverage: {coverage['completed_cells']}/{expected} cells; "
          f"{'COMPLETE' if coverage['complete'] else 'INCOMPLETE — inspect failures.json'}. "
          f"{'Training-only smoke, not scientific evidence.' if config['SMOKE'] else ''}", flush=True)
    return {"run_dir": str(run_dir), "results": results, "summary": summary,
            "paired_deltas": paired, "comparison": comparison}
'''


In [ ]:
# Materialize this notebook's embedded code, not code from the current directory.
import hashlib
import importlib
import json
from pathlib import Path
import sys
import tempfile

BUNDLE_SHA256 = hashlib.sha256(json.dumps(BUNDLE_SOURCES, sort_keys=True).encode()).hexdigest()
BUNDLE_ROOT = Path(tempfile.mkdtemp(prefix="ugo_notebook_"))
PACKAGE_ROOT = BUNDLE_ROOT / "portable_ugo"
PACKAGE_ROOT.mkdir()
for relative, source in BUNDLE_SOURCES.items():
    destination = PACKAGE_ROOT / relative
    destination.write_text(source, encoding="utf-8")
# Rerunning cells replaces only this notebook's own package namespace.
for name in list(sys.modules):
    if name == "portable_ugo" or name.startswith("portable_ugo."):
        del sys.modules[name]
sys.path.insert(0, str(BUNDLE_ROOT))
importlib.invalidate_caches()
from portable_ugo.benchmark import run_benchmark, select_catalogue, show_catalogue
print("Bundled pipeline SHA256:", BUNDLE_SHA256)
print("Private runtime package:", PACKAGE_ROOT)


In [ ]:
# Fixed, explicit CLIMB / raw OpenML catalogue.
CATALOGUE = [{'number': 1, 'openml_id': 976, 'name': 'JapaneseVowels', 'imbalance_type': 'medium'},
 {'number': 2, 'openml_id': 43893, 'name': 'ibm-employee-attrition', 'imbalance_type': 'medium'},
 {'number': 3, 'openml_id': 45560, 'name': 'online-shoppers-intention', 'imbalance_type': 'medium'},
 {'number': 4, 'openml_id': 1067, 'name': 'kc1', 'imbalance_type': 'medium'},
 {'number': 5, 'openml_id': 1506, 'name': 'thoracic-surgery', 'imbalance_type': 'medium'},
 {'number': 6, 'openml_id': 44232, 'name': 'UCI_churn', 'imbalance_type': 'medium'},
 {'number': 7, 'openml_id': 949, 'name': 'arsenic-female-bladder', 'imbalance_type': 'medium'},
 {'number': 8, 'openml_id': 1049, 'name': 'pc4', 'imbalance_type': 'medium'},
 {'number': 9, 'openml_id': 1558, 'name': 'bank-marketing', 'imbalance_type': 'medium'},
 {'number': 10, 'openml_id': 43903, 'name': 'Diabetes-130-Hospitals_(Fairlearn)', 'imbalance_type': 'medium'},
 {'number': 11, 'openml_id': 1019, 'name': 'pendigits', 'imbalance_type': 'medium'},
 {'number': 12, 'openml_id': 1050, 'name': 'pc3', 'imbalance_type': 'medium'},
 {'number': 13, 'openml_id': 1021, 'name': 'page-blocks-bin', 'imbalance_type': 'medium'},
 {'number': 14, 'openml_id': 980, 'name': 'optdigits', 'imbalance_type': 'medium'},
 {'number': 15, 'openml_id': 1020, 'name': 'mfeat-karhunen', 'imbalance_type': 'medium'},
 {'number': 16, 'openml_id': 971, 'name': 'mfeat-fourier', 'imbalance_type': 'medium'},
 {'number': 17, 'openml_id': 995, 'name': 'mfeat-zernike', 'imbalance_type': 'medium'},
 {'number': 18, 'openml_id': 45558, 'name': 'Pulsar-Dataset-HTRU2', 'imbalance_type': 'medium'},
 {'number': 19, 'openml_id': 1016, 'name': 'vowel', 'imbalance_type': 'high'},
 {'number': 20, 'openml_id': 1068, 'name': 'pc1', 'imbalance_type': 'high'},
 {'number': 21, 'openml_id': 45562, 'name': 'seismic-bumps', 'imbalance_type': 'high'},
 {'number': 22, 'openml_id': 1487, 'name': 'ozone-level-8hr', 'imbalance_type': 'high'},
 {'number': 23, 'openml_id': 41946, 'name': 'Sick_numeric', 'imbalance_type': 'high'},
 {'number': 24, 'openml_id': 46281, 'name': 'insurance_company', 'imbalance_type': 'high'},
 {'number': 25, 'openml_id': 40983, 'name': 'wilt', 'imbalance_type': 'high'},
 {'number': 26, 'openml_id': 1217, 'name': 'Click_prediction_small', 'imbalance_type': 'high'},
 {'number': 27, 'openml_id': 977, 'name': 'letter', 'imbalance_type': 'high'},
 {'number': 28, 'openml_id': 310, 'name': 'mammography', 'imbalance_type': 'high'},
 {'number': 29, 'openml_id': 40713, 'name': 'dis', 'imbalance_type': 'extreme'},
 {'number': 30, 'openml_id': 40900, 'name': 'Satellite', 'imbalance_type': 'extreme'},
 {'number': 31, 'openml_id': 43, 'name': 'haberman', 'imbalance_type': 'low'},
 {'number': 32, 'openml_id': 1464, 'name': 'blood-transfusion-service-center', 'imbalance_type': 'low'}]
from IPython.display import display
display(show_catalogue(CATALOGUE).assign(default_run=lambda rows: ~rows["number"].isin(CONFIG["EXCLUDED_DATASETS"])))
SELECTED_DATASETS = select_catalogue(CATALOGUE, CONFIG["DATASET_RANGE"], excluded_numbers=CONFIG["EXCLUDED_DATASETS"])
print("Selected:", [(row["number"], row["name"]) for row in SELECTED_DATASETS])
print("Excluded dataset IDs:", CONFIG["EXCLUDED_DATASETS"])


## Execute the selected range

This runs actual model fitting. The selected validation threshold is frozen before each final outer refit. Both scientific and smoke outputs are labelled, and all failures are retained in the run directory. Do not treat a partial run or smoke result as a complete study.


In [ ]:
RUN_RESULTS = run_benchmark(CONFIG, CATALOGUE)


In [ ]:
import pandas as pd
from portable_ugo.benchmark import METHODS

print("Results directory:", RUN_RESULTS["run_dir"])
labels = {"baseline": "No oversampling", "original_ugo": "Original UGO",
          "u2_flow_gate": "U2 + Flow gate", "smote": "SMOTE",
          "kmeans_smote": "KMeansSMOTE", "rwo": "RWO"}
columns = {f"{method}_minority_f1_mean": labels[method] for method in METHODS}
view = RUN_RESULTS["comparison"][["dataset_number", "dataset", "classifier", "status",
                                  *columns]].rename(columns=columns)
print("Mean minority F1. INCOMPLETE rows are not full comparisons.")
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(view)
print("All F1 standard deviations and seed coverage:", Path(RUN_RESULTS["run_dir"]) / "comparison.csv")
print("All metrics:", Path(RUN_RESULTS["run_dir"]) / "summary.csv")
print("All 15 matched method pairs:", Path(RUN_RESULTS["run_dir"]) / "paired_deltas.csv")
